In [225]:
import pandas as pd
from unidecode import unidecode
import geopandas as gpd
import pydeck as pdk
import json
import webbrowser
from pathlib import Path
from datetime import datetime
import os
from functools import reduce
import duckdb
import numpy as np        
import folium
from folium import plugins
hoy = datetime.today()


usuario = os.getlogin()
current_date = datetime.now().strftime("%d-%m-%y")

In [226]:
base_tode = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR/División de Procesamiento de información - Repositorio de Datos/Productividad/conteos con ece/todes_consolidadas.parquet")
base_metas = pd.read_excel(rf"C:\Users\{usuario}\IMSS-BIENESTAR/División de Procesamiento de información - Repositorio de Datos\Productividad\Metas\2026\Metas de productividad por unidad medica 2026.xlsx")
clues = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")


In [227]:
unicos = base_tode['tipo_consulta'].unique()
unicos

array(['general', 'especialidad', 'qx', 'egresos'], dtype=object)

In [228]:
consu= base_tode[base_tode['anio_insert'] == "2026"]

In [229]:
consu = consu[
    consu['tipo_consulta'].isin(['general', 'especialidad', 'qx'])
]

In [230]:
consu['tipo_consulta'] = consu['tipo_consulta'].replace({
    'qx': 'cirugia'
})

In [231]:
cirugia = consu.loc[
    consu['tipo_consulta'] == 'cirugia',
    'procedimientos'
].sum()

consulta = consu.loc[
    consu['tipo_consulta'].isin(['general', 'especialidad']),
    'procedimientos'
].sum()
general = consu.loc[
    consu['tipo_consulta'] == 'general',
    'procedimientos'
].sum()
especialidad = consu.loc[
    consu['tipo_consulta'] == 'especialidad',
    'procedimientos'
].sum()
conteo_consu = pd.DataFrame({
    'cirugia': [cirugia],
    'consulta': [consulta],
    'general': [general],
    'especialidad': [especialidad]
})

In [232]:
conteo_consu.columns

Index(['cirugia', 'consulta', 'general', 'especialidad'], dtype='object')

In [233]:
query = """
WITH base AS (
    SELECT
        clues AS clues_imb,

        SUM(CASE WHEN tipo_consulta = 'general' THEN procedimientos ELSE 0 END) AS general,
        SUM(CASE WHEN tipo_consulta = 'qx' THEN procedimientos ELSE 0 END) AS qx,
        SUM(CASE WHEN tipo_consulta = 'especialidad' THEN procedimientos ELSE 0 END) AS especialidad,
        SUM(CASE WHEN tipo_consulta = 'egresos' THEN procedimientos ELSE 0 END) AS egresos

    FROM base_tode
    WHERE anio_insert = 2026
    GROUP BY clues
),

metas AS (
    SELECT
        entidad,
        clues_imb,
        estatus_de_operacion,
        nombre_de_la_unidad,
        nivel_atencion,
        meta_general_anual,
        meta_especialidad_anual,
        meta_cirugia_anual,
        meta_egresos_anual
    FROM base_metas
)

SELECT
    m.clues_imb,
    m.entidad,
    m.estatus_de_operacion,
    m.nombre_de_la_unidad,
    m.nivel_atencion,

    b.general,
    b.qx,
    b.especialidad,
    b.egresos,

    m.meta_general_anual,
    m.meta_especialidad_anual,
    m.meta_cirugia_anual,
    m.meta_egresos_anual

FROM metas m
LEFT JOIN base b
    ON m.clues_imb = b.clues_imb
"""
base = duckdb.query(query).to_df()
base = base.rename(columns={
    "qx": "cirugias",})

In [234]:
cols = [
    'clues_imb', 'entidad', 'estatus_de_operacion', 'nombre_de_la_unidad',
    'nivel_atencion', 'general', 'cirugias', 'especialidad', 'egresos',
    'meta_general_anual', 'meta_especialidad_anual',
    'meta_cirugia_anual', 'meta_egresos_anual'
]

base = base[cols].copy()


In [235]:
base_metas

,entidad,clues_imb,categoria_gerencial,estatus_de_operacion,nombre_de_la_unidad,nivel_atencion,meta_general_anual,meta_especialidad_anual,meta_cirugia_anual,meta_egresos_anual
0,BAJA CALIFORNIA,BCIMB000010,Generales 100-149 c,EN OPERACION,HOSPITAL GENERAL DE ENSENADA,SEGUNDO NIVEL,0,59157,6095,7305
1,BAJA CALIFORNIA,BCIMB000022,Unidades moviles,EN OPERACION,UNIDAD MÓVIL NO. 7,PRIMER NIVEL,1899,0,0,0
2,BAJA CALIFORNIA,BCIMB000034,Unidades moviles,EN OPERACION,UNIDAD MÓVIL NO. 8,PRIMER NIVEL,1899,0,0,0
3,BAJA CALIFORNIA,BCIMB000046,Unidades moviles,EN OPERACION,UNIDAD MÓVIL 9,PRIMER NIVEL,1899,0,0,0
4,BAJA CALIFORNIA,BCIMB000051,1-2 nucleos,EN OPERACION,COLONIA LOMA LINDA,PRIMER NIVEL,3708,0,0,0
...,...,...,...,...,...,...,...,...,...,...
10573,ZACATECAS,ZSIMB002621,Unidades moviles,EN OPERACION,CARAVANA DE LA SALUD TIPO 0 CIENEGUILLA (NORIA...,NO APLICA,1694,0,0,0
10574,ZACATECAS,ZSIMB002633,Unidades moviles,EN OPERACION,CARAVANA DE LA SALUD TIPO 0 LA VILLITA,NO APLICA,1694,0,0,0
10575,ZACATECAS,ZSIMB002650,6-12 nucleos,EN OPERACION,CENTRO DE SALUD JEREZ,PRIMER NIVEL,12428,0,0,0
10576,CHIAPAS,CSIMB003902,Servicios ampliados,EN OPERACION,CLINICA PARA LA ATENCION DE PARTO HUMANIZADO S...,SEGUNDO NIVEL,0,2301,0,0


In [236]:
conteo_consultas = (
    base
    .agg({
        'cirugias': 'sum',
        'general': 'sum',
        'especialidad': 'sum'
    })
)

conteo_consultas = pd.DataFrame({
    'cirugia': [conteo_consultas['cirugias']],
    'consulta': [conteo_consultas['general'] + conteo_consultas['especialidad']]
})

In [237]:
conteo_consultas

,cirugia,consulta
0,356106.0,22725087.0


In [238]:

#  consultas según nivel de atención
base['consultas'] = np.where(
    base['nivel_atencion'].isin(['SEGUNDO NIVEL', 'TERCER NIVEL']),
    base['cirugias'],
    base[['general','especialidad']].sum(axis=1)
)

#  meta total (siempre suma de todas las metas)
base['meta_total'] = (
    base['meta_general_anual']
    + base['meta_especialidad_anual']
    + base['meta_cirugia_anual']
    + base['meta_egresos_anual']
)

In [239]:
cols_base =['clues_imb', 'entidad', 'nombre_de_la_unidad',
       'nivel_atencion', 'cirugias','meta_cirugia_anual','consultas', 'meta_total']
base = base[cols_base].copy()   

In [240]:
base = base.merge(
    clues[['clues_imb', 'latitud', 'longitud']], 
    on='clues_imb',
    how='left'
)

In [241]:
columnas_finales = [
 'clues_imb', 'entidad', 'nombre_de_la_unidad', 'nivel_atencion',
       'cirugias', 'meta_cirugia_anual', 'consultas', 'meta_total', 'latitud',
       'longitud'
]
base = base.drop_duplicates(subset=columnas_finales, keep="first")

In [242]:
from datetime import timedelta

dias_desde_miercoles = (hoy.weekday() - 2) % 7
if dias_desde_miercoles == 0:
    dias_desde_miercoles = 7
else:
    dias_desde_miercoles += 7
ultimo_miercoles = hoy - timedelta(days=dias_desde_miercoles)

dias_transcurridos = ultimo_miercoles.timetuple().tm_yday
dias_del_anio = 365 + int(ultimo_miercoles.year % 4 == 0 and (ultimo_miercoles.year % 100 != 0 or ultimo_miercoles.year % 400 == 0))

base['pct_tiempo'] = dias_transcurridos / dias_del_anio
base['nivel_atencion'] = base['nivel_atencion'].str.strip().str.upper()
mask = base['nivel_atencion'].isin(['SEGUNDO NIVEL', 'TERCER NIVEL'])

base.loc[mask, 'meta_esperada_ciru'] = (
    base.loc[mask, 'meta_cirugia_anual'] * base.loc[mask, 'pct_tiempo']
 )
mask = base['nivel_atencion'].isin(['SEGUNDO NIVEL', 'TERCER NIVEL'])

base['pct_cirugia'] = np.where(
    mask,
    base['cirugias'] / base['meta_cirugia_anual'] * 100,
    np.nan
)

In [243]:
mask_consulta = ~base['nivel_atencion'].isin(['SEGUNDO NIVEL', 'TERCER NIVEL'])

base.loc[mask_consulta, 'meta_esperada_consulta'] = (
    base.loc[mask_consulta, 'meta_total'] * base.loc[mask_consulta, 'pct_tiempo']
)

In [244]:
base['pct_consulta'] = np.where(
    mask_consulta,
    base['consultas'] / base['meta_total'] * 100,
    np.nan
)

In [245]:
import numpy as np

mask_cirugia = base['nivel_atencion'].isin(['SEGUNDO NIVEL', 'TERCER NIVEL'])
mask_consulta = ~mask_cirugia

base['pct_general'] = np.where(
    mask_cirugia,
    base['cirugias'] / base['meta_cirugia_anual'] * 100,
    base['consultas'] / base['meta_total'] * 100
)

In [246]:
base['pct_general'] = base['pct_general'].round(1)

In [247]:
base.columns

Index(['clues_imb', 'entidad', 'nombre_de_la_unidad', 'nivel_atencion',
       'cirugias', 'meta_cirugia_anual', 'consultas', 'meta_total', 'latitud',
       'longitud', 'pct_tiempo', 'meta_esperada_ciru', 'pct_cirugia',
       'meta_esperada_consulta', 'pct_consulta', 'pct_general'],
      dtype='object')

In [248]:
suma_cirugias = base['cirugias'].sum()
suma_consultas = base['consultas'].sum()

print(f'Cirugías: {suma_cirugias}')
print(f'Consultas: {suma_consultas}')

Cirugías: 356106.0
Consultas: 18485844.0


In [249]:
base

,clues_imb,entidad,nombre_de_la_unidad,nivel_atencion,cirugias,meta_cirugia_anual,consultas,meta_total,latitud,longitud,pct_tiempo,meta_esperada_ciru,pct_cirugia,meta_esperada_consulta,pct_consulta,pct_general
0,OCIMB007712,OAXACA,SANTA CATARINA ADEQUEZ,PRIMER NIVEL,0.0,0,299.0,1661,17.434400,-97.144400,0.421918,NaN,NaN,700.805479,18.001204,18.0
1,SPIMB002125,SAN LUIS POTOSI,HOSPITAL GENERAL DE MATEHUALA,SEGUNDO NIVEL,865.0,2079,865.0,18582,23.669090,-100.653531,0.421918,877.167123,41.606542,NaN,NaN,41.6
2,PLIMB004271,PUEBLA,CENTRO DE SALUD ATLA,PRIMER NIVEL,0.0,0,2455.0,5876,20.273309,-98.127343,0.421918,NaN,NaN,2479.189041,41.780123,41.8
3,MCIMB005966,MEXICO,SAN FRANCISCO TEPEXOXUCA,PRIMER NIVEL,0.0,0,1542.0,4761,19.064318,-99.546272,0.421918,NaN,NaN,2008.750685,32.388154,32.4
4,OCIMB007374,OAXACA,R 01 PEÑA COLORADA,PRIMER NIVEL,0.0,0,680.0,1928,17.477778,-97.794167,0.421918,NaN,NaN,813.457534,35.269710,35.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10573,SLIMB002406,SINALOA,UNIDAD MÓVIL DE LA MUJER,PRIMER NIVEL,NaN,0,0.0,1810,24.810092,-107.383914,0.421918,NaN,NaN,763.671233,0.000000,0.0
10574,TCSSA017222,TABASCO,None,PRIMER NIVEL,NaN,0,0.0,1624,NaN,NaN,0.421918,NaN,NaN,685.194521,0.000000,0.0
10575,TSIMB003676,TAMAULIPAS,CENTRO DE SALUD PALO SOLO,PRIMER NIVEL,NaN,0,0.0,2920,25.085069,-97.918022,0.421918,NaN,NaN,1232.000000,0.000000,0.0
10576,VZIMB004990,VERACRUZ,URSULO GALVAN (LAS CHARCAS),PRIMER NIVEL,NaN,0,0.0,2911,18.548800,-96.056600,0.421918,NaN,NaN,1228.202740,0.000000,0.0


## mapa

In [250]:

def asignar_macro_categoria(nivel_atencion):
    if nivel_atencion in ["SEGUNDO NIVEL", "TERCER NIVEL"]:
        return "cirugias"
    else:
        return "consultas"

semaforo_colores = {
    "rojo": "#D41111",
    "amarillo": "#F1D54A",
    "verde claro": "#88A91E",
    "verde fuerte": "#0D5D2A"
}

PCT_ANIO_PORCENTAJE = 0

def obtener_color_semaforo(avance):
    global PCT_ANIO_PORCENTAJE
    
    if pd.isna(avance) or np.isinf(avance):
        return semaforo_colores["rojo"]
    
    if PCT_ANIO_PORCENTAJE == 0:
        if avance >= 95:
            return semaforo_colores["verde fuerte"]
        elif avance >= 80:
            return semaforo_colores["verde claro"]
        elif avance >= 60:
            return semaforo_colores["amarillo"]
        else:
            return semaforo_colores["rojo"]
    
    verde_fuerte_threshold = PCT_ANIO_PORCENTAJE * 0.95
    verde_claro_threshold = PCT_ANIO_PORCENTAJE * 0.80
    amarillo_threshold = PCT_ANIO_PORCENTAJE * 0.60
    
    if avance >= verde_fuerte_threshold:
        return semaforo_colores["verde fuerte"]
    elif avance >= verde_claro_threshold:
        return semaforo_colores["verde claro"]
    elif avance >= amarillo_threshold:
        return semaforo_colores["amarillo"]
    else:
        return semaforo_colores["rojo"]

def obtener_estado_semaforo(avance):
    global PCT_ANIO_PORCENTAJE
    
    if pd.isna(avance) or np.isinf(avance):
        return "Sin datos validos"
    
    verde_fuerte_threshold = PCT_ANIO_PORCENTAJE * 0.95
    verde_claro_threshold = PCT_ANIO_PORCENTAJE * 0.80
    amarillo_threshold = PCT_ANIO_PORCENTAJE * 0.60
    
    if avance >= verde_fuerte_threshold:
        return f"Verde Fuerte (Avance >= {verde_fuerte_threshold:.1f}%)"
    elif avance >= verde_claro_threshold:
        return f"Verde Claro (Avance >= {verde_claro_threshold:.1f}%)"
    elif avance >= amarillo_threshold:
        return f"Amarillo (Avance >= {amarillo_threshold:.1f}%)"
    else:
        return f"Rojo (Avance < {amarillo_threshold:.1f}%)"

In [251]:

df_plot = base.copy()

df_plot["latitud"] = pd.to_numeric(df_plot["latitud"], errors="coerce")
df_plot["longitud"] = pd.to_numeric(df_plot["longitud"], errors="coerce")
df_plot["pct_general"] = pd.to_numeric(df_plot["pct_general"], errors="coerce")
df_plot["cirugias"] = pd.to_numeric(df_plot["cirugias"], errors="coerce").fillna(0)
df_plot["consultas"] = pd.to_numeric(df_plot["consultas"], errors="coerce").fillna(0)
df_plot["meta_cirugia_anual"] = pd.to_numeric(df_plot["meta_cirugia_anual"], errors="coerce").fillna(1)
df_plot["meta_total"] = pd.to_numeric(df_plot["meta_total"], errors="coerce").fillna(1)
df_plot["meta_esperada_ciru"] = pd.to_numeric(df_plot["meta_esperada_ciru"], errors="coerce").fillna(1)
df_plot["meta_esperada_consulta"] = pd.to_numeric(df_plot["meta_esperada_consulta"], errors="coerce").fillna(1)

df_plot["pct_general"] = df_plot["pct_general"].replace([np.inf, -np.inf], np.nan)

df_plot = df_plot.dropna(subset=["latitud", "longitud", "pct_general"]).copy()
df_plot["pct_general"] = df_plot["pct_general"].astype(float)

In [252]:
if "pct_tiempo" in df_plot.columns and len(df_plot) > 0:
    pct_tiempo_decimal = pd.to_numeric(df_plot["pct_tiempo"].iloc[0], errors="coerce")
    if pd.isna(pct_tiempo_decimal) or np.isinf(pct_tiempo_decimal):
        pct_tiempo_decimal = 0
else:
    pct_tiempo_decimal = 0

pct_dia_esperado = pct_tiempo_decimal * 100
PCT_ANIO_PORCENTAJE = pct_dia_esperado

In [253]:
PCT_ANIO_PORCENTAJE

np.float64(42.19178082191781)

In [254]:


df_plot["pct_dia"] = pct_dia_esperado
df_plot["pct_dia_fmt"] = df_plot["pct_dia"].map(lambda x: f"{x:.1f}%")

df_plot["macro_key"] = df_plot["nivel_atencion"].apply(asignar_macro_categoria)

df_plot["color_hex"] = df_plot["pct_general"].apply(obtener_color_semaforo)

df_plot["pct_general_fmt"] = df_plot["pct_general"].map(
    lambda x: f"{x:.1f}%" if not pd.isna(x) else "0.0%"
)

df_plot["estado_semaforo"] = df_plot["pct_general"].apply(obtener_estado_semaforo)

In [255]:


def formatear_metricas(row):
    if row["nivel_atencion"] in ["SEGUNDO NIVEL", "TERCER NIVEL"]:
        realizado = row.get("cirugias", 0)
        if pd.isna(realizado) or np.isinf(realizado):
            realizado = 0
        meta_anual = row.get("meta_cirugia_anual", 1)
        if pd.isna(meta_anual) or np.isinf(meta_anual):
            meta_anual = 1
        meta_esperada = row.get("meta_esperada_ciru", 1)
        if pd.isna(meta_esperada) or np.isinf(meta_esperada):
            meta_esperada = 1
        tipo = "cirugias"
    else:
        realizado = row.get("consultas", 0)
        if pd.isna(realizado) or np.isinf(realizado):
            realizado = 0
        meta_anual = row.get("meta_total", 1)
        if pd.isna(meta_anual) or np.isinf(meta_anual):
            meta_anual = 1
        meta_esperada = row.get("meta_esperada_consulta", 1)
        if pd.isna(meta_esperada) or np.isinf(meta_esperada):
            meta_esperada = 1
        tipo = "consultas"
    
    return pd.Series({
        "realizado_fmt": f"{int(realizado):,}",
        "meta_anual_fmt": f"{int(meta_anual):,}",
        "meta_esperada_fmt": f"{int(meta_esperada):,}",
        "tipo_indicador": tipo
    })

df_plot[
    ["realizado_fmt", "meta_anual_fmt", "meta_esperada_fmt", "tipo_indicador"]
] = df_plot.apply(formatear_metricas, axis=1)


In [256]:
# entidades = df_validos["entidad"].unique()

In [257]:
df_validos = df_plot[df_plot["pct_general"].notna()].copy()
df_validos = df_validos[~np.isinf(df_validos["pct_general"])].copy()

df_sin_ceros = df_validos[df_validos["pct_general"] > 0].copy()

df_cirugias = df_validos[df_validos["macro_key"] == "cirugias"].copy()
df_consultas = df_validos[df_validos["macro_key"] == "consultas"].copy()

total_unidades_activas = len(df_validos)

unidades_cirugias = len(df_sin_ceros[df_sin_ceros["macro_key"] == "cirugias"])
unidades_consultas = len(df_sin_ceros[df_sin_ceros["macro_key"] == "consultas"])

if len(df_cirugias) > 0:
    total_cirugias = df_cirugias["cirugias"].sum()
    total_cirugias = total_cirugias if not pd.isna(total_cirugias) else 0
    meta_cirugias_total = df_cirugias["meta_cirugia_anual"].sum()
    meta_cirugias_total = meta_cirugias_total if not pd.isna(meta_cirugias_total) else 0
else:
    total_cirugias = 0
    meta_cirugias_total = 0

if len(df_consultas) > 0:
    total_consultas = df_consultas["consultas"].sum()
    total_consultas = total_consultas if not pd.isna(total_consultas) else 0
    meta_consultas_total = df_consultas["meta_total"].sum()
    meta_consultas_total = meta_consultas_total if not pd.isna(meta_consultas_total) else 0
else:
    total_consultas = 0
    meta_consultas_total = 0

total_realizado = total_cirugias + total_consultas
meta_anual_total = meta_cirugias_total + meta_consultas_total

if meta_anual_total > 0:
    avg_avance_general = total_realizado / meta_anual_total * 100
    avg_avance_general = avg_avance_general if not pd.isna(avg_avance_general) else 0
else:
    avg_avance_general = 0

if meta_cirugias_total > 0:
    avg_avance_cirugias = total_cirugias / meta_cirugias_total * 100
    avg_avance_cirugias = avg_avance_cirugias if not pd.isna(avg_avance_cirugias) else 0
else:
    avg_avance_cirugias = 0

if meta_consultas_total > 0:
    avg_avance_consultas = total_consultas / meta_consultas_total * 100
    avg_avance_consultas = avg_avance_consultas if not pd.isna(avg_avance_consultas) else 0
else:
    avg_avance_consultas = 0

In [258]:
meta_consultas_total

np.int64(52261856)

In [259]:
avg_avance_consultas


np.float64(34.69104885980322)

In [260]:

unidades_data = []
for _, row in df_validos.iterrows():
    unidades_data.append({
        "clues": str(row["clues_imb"]),
        "nombre": str(row["nombre_de_la_unidad"]),
        "lat": float(row["latitud"]),
        "lng": float(row["longitud"]),
        "avance": row["pct_general_fmt"],
        "estado": row["estado_semaforo"]
    })

In [261]:
# # mapa jsjsjsjs
# mapa_center_lat = df_validos["latitud"].mean()
# mapa_center_lon = df_validos["longitud"].mean()

# m = folium.Map(  # puro pinche folium y no mmdas jsjsjsjsjs
#     location=[mapa_center_lat, mapa_center_lon],
#     zoom_start=6,
#     tiles="CartoDB dark_matter",
#     control_scale=True
# )

## html

In [262]:
# ============================================================
# PREPARACION DE DATOS PARA 3D (USANDO TU SEMÁFORO)
# ============================================================

df_3d = df_validos.copy()

# Calcular el porcentaje del año transcurrido (como ya lo haces)
if "pct_tiempo" in df_3d.columns and len(df_3d) > 0:
    pct_tiempo_decimal = pd.to_numeric(df_3d["pct_tiempo"].iloc[0], errors="coerce")
    if pd.isna(pct_tiempo_decimal) or np.isinf(pct_tiempo_decimal):
        pct_tiempo_decimal = 0
else:
    pct_tiempo_decimal = 0

pct_dia_esperado = pct_tiempo_decimal * 100

# Actualizar la variable global para tus funciones
PCT_ANIO_PORCENTAJE = pct_dia_esperado

# Procesar los datos numéricos
if df_3d["pct_general"].dtype == "object":
    df_3d["pct_general_num"] = pd.to_numeric(df_3d["pct_general"].str.rstrip('%'), errors='coerce') / 100
else:
    if df_3d["pct_general"].max() > 1:
        df_3d["pct_general_num"] = df_3d["pct_general"] / 100
    else:
        df_3d["pct_general_num"] = df_3d["pct_general"]

df_3d["pct_general_num"] = df_3d["pct_general_num"].fillna(0).clip(0, 1)

# Altura de las columnas
df_3d["altura"] = 500 + (df_3d["pct_general_num"] * 20000)

# ========== USAR TU FUNCIÓN DE SEMÁFORO ==========
def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

# Aplicar tu función obtener_color_semaforo
df_3d["color_semaforo_hex"] = df_3d["pct_general"].apply(obtener_color_semaforo)

# Convertir a RGB para deck.gl
df_3d[["r", "g", "b"]] = df_3d["color_semaforo_hex"].apply(
    lambda x: pd.Series(hex_to_rgb(x))
)

# Calcular los umbrales para mostrar en el HTML
verde_fuerte_umbral = PCT_ANIO_PORCENTAJE * 0.95
verde_claro_umbral = PCT_ANIO_PORCENTAJE * 0.80
amarillo_umbral = PCT_ANIO_PORCENTAJE * 0.60

# ========== EXTRACCIÓN DE VOLÚMENES CON DETALLE ==========
volumen_cirugias = pd.to_numeric(conteo_consultas.loc[0, "cirugia"], errors="coerce")
volumen_consultas = pd.to_numeric(conteo_consultas.loc[0, "consulta"], errors="coerce")

# Tomar desglose de general y especialidad de conteo_consu
consulta_general = pd.to_numeric(conteo_consu.loc[0, "general"], errors="coerce")
consulta_especialidad = pd.to_numeric(conteo_consu.loc[0, "especialidad"], errors="coerce")

volumen_cirugias = 0 if pd.isna(volumen_cirugias) else volumen_cirugias
volumen_consultas = 0 if pd.isna(volumen_consultas) else volumen_consultas
consulta_general = 0 if pd.isna(consulta_general) else consulta_general
consulta_especialidad = 0 if pd.isna(consulta_especialidad) else consulta_especialidad
volumen_realizado = volumen_cirugias + volumen_consultas

# Datos para el buscador
unidades_data = []
for _, row in df_3d.iterrows():
    unidades_data.append({
        "clues": row['clues_imb'],
        "nombre": row['nombre_de_la_unidad'],
        "lat": row['latitud'],
        "lng": row['longitud'],
        "avance": row['pct_general_fmt'],
        "entidad": row['entidad'],
        "nivel": row['nivel_atencion'],
        "tipo": row['tipo_indicador'],
        "realizado": row['realizado_fmt'],
        "meta_anual": row['meta_anual_fmt'],
        "meta_esperada": row['meta_esperada_fmt'],
        "pct_dia_fmt": row['pct_dia_fmt'],
        "color_hex": row['color_semaforo_hex'],
        "estado_semaforo": row['estado_semaforo'],
        "nombre_de_la_unidad": row['nombre_de_la_unidad'],
        "altura": row['altura'],
        "r": row['r'],
        "g": row['g'],
        "b": row['b']
    })
unidades_data_json = json.dumps(unidades_data, ensure_ascii=False)

# Calcular centro del mapa
lat_centro = df_3d["latitud"].mean()
lon_centro = df_3d["longitud"].mean()

# ============================================================
# GENERAR HTML CON DOS MODOS (3D y 2D optimizado)
# ============================================================

html_dual = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<title>Hospital Dashboard - Modo 3D y 2D para Rutas</title>

<script src="https://unpkg.com/deck.gl@latest/dist.min.js"></script>
<script src="https://unpkg.com/maplibre-gl@latest/dist/maplibre-gl.js"></script>
<link href="https://unpkg.com/maplibre-gl@latest/dist/maplibre-gl.css" rel="stylesheet"/>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://unpkg.com/leaflet-routing-machine@latest/dist/leaflet-routing-machine.js"></script>
<link rel="stylesheet" href="https://unpkg.com/leaflet-routing-machine@latest/dist/leaflet-routing-machine.css" />

<style>
html, body {{
    margin: 0;
    width: 100%;
    height: 100%;
    font-family: 'Segoe UI', Arial, sans-serif;
    background: #0a0a0a;
}}

#map-3d {{
    width: 100%;
    height: 100%;
    background: #0d0d0d;
    display: block;
}}

#map-2d {{
    width: 100%;
    height: 100%;
    background: #0d0d0d;
    display: none;
}}

/* Ocultar la leyenda de Leaflet */
.leaflet-control-attribution {{
    display: none !important;
}}

.header {{
    position: fixed;
    top: 0;
    left: 0;
    right: 0;
    z-index: 10000;
    background: #1E5B4F;
    color: white;
    padding: 12px 25px;
    font-size: 18px;
    font-weight: bold;
    text-align: center;
    box-shadow: 0 2px 10px rgba(0,0,0,0.5);
    pointer-events: none;
    border-bottom: 1px solid rgba(165,127,44,0.3);
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 15px;
}}

.header img {{
    height: 40px;
    width: auto;
    filter: brightness(0) invert(1);
}}

.btn-toggle {{
    position: fixed;
    top: 80px;
    right: 20px;
    z-index: 10001;
    background: linear-gradient(135deg, #A57F2C, #1E5B4F);
    color: white;
    border: none;
    padding: 12px 24px;
    border-radius: 30px;
    font-size: 14px;
    font-weight: bold;
    cursor: pointer;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-shadow: 0 4px 15px rgba(0,0,0,0.3);
    transition: all 0.3s;
    pointer-events: auto;
}}

.btn-toggle:hover {{
    transform: translateY(-2px);
    box-shadow: 0 6px 20px rgba(165,127,44,0.4);
}}

.kpi-container {{
    position: fixed;
    top: 70px;
    left: 0;
    right: 0;
    z-index: 10000;
    display: grid;
    grid-template-columns: repeat(5, 1fr);
    gap: 12px;
    padding: 12px 20px;
    pointer-events: none;
}}

.kpi-card {{
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    padding: 10px 15px;
    border-radius: 10px;
    border: 1px solid rgba(165,127,44,0.3);
    pointer-events: auto;
    box-shadow: 0 2px 5px rgba(0,0,0,0.3);
}}

.kpi-label {{
    font-size: 10px;
    color: #A57F2C;
    text-transform: uppercase;
    letter-spacing: 1px;
}}

.kpi-value {{
    font-size: 24px;
    font-weight: bold;
    color: white;
}}

.kpi-sub {{
    font-size: 9px;
    color: #888;
}}

.dashboard-btn {{
    position: fixed;
    top: 380px;
    left: 20px;
    z-index: 10001;
}}

.dashboard-btn button {{
    background: linear-gradient(135deg, #A57F2C, #1E5B4F);
    color: white;
    border: none;
    padding: 8px 18px;
    border-radius: 25px;
    font-size: 12px;
    font-weight: bold;
    cursor: pointer;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-shadow: 0 2px 5px rgba(0,0,0,0.3);
    transition: all 0.3s;
    pointer-events: auto;
}}

.dashboard-btn button:hover {{
    transform: translateY(-2px);
    box-shadow: 0 4px 10px rgba(165,127,44,0.3);
}}

.buscador {{
    position: fixed;
    top: 200px;
    left: 20px;
    z-index: 10001;
    width: 320px;
}}

.buscador-box {{
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    border-radius: 12px;
    padding: 12px;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
}}

.buscador-label {{
    color: #A57F2C;
    font-size: 10px;
    text-transform: uppercase;
    margin-bottom: 8px;
    letter-spacing: 1px;
}}

.buscador-input {{
    width: 100%;
    padding: 10px 15px;
    background: rgba(20,20,20,0.9);
    border: 1px solid rgba(165,127,44,0.3);
    border-radius: 8px;
    color: white;
    font-size: 13px;
    outline: none;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-sizing: border-box;
}}

.buscador-input:focus {{
    border-color: #A57F2C;
    box-shadow: 0 0 5px rgba(165,127,44,0.5);
}}

.resultados {{
    background: rgba(10,10,10,0.95);
    border-radius: 12px;
    margin-top: 8px;
    max-height: 350px;
    overflow-y: auto;
    display: none;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
}}

.resultado-item {{
    padding: 12px 15px;
    cursor: pointer;
    border-bottom: 1px solid rgba(165,127,44,0.1);
    font-family: 'Segoe UI', Arial, sans-serif;
    transition: all 0.2s;
}}

.resultado-item:hover {{
    background: rgba(165,127,44,0.15);
    border-left: 3px solid #A57F2C;
}}

.rutas-panel {{
    position: fixed;
    bottom: 20px;
    left: 20px;
    z-index: 10001;
    background: rgba(10,10,10,0.95);
    backdrop-filter: blur(10px);
    border-radius: 12px;
    min-width: 280px;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    pointer-events: auto;
    padding: 12px;
    display: none;
}}

.rutas-panel h4 {{
    margin: 0 0 10px 0;
    color: #A57F2C;
    font-size: 13px;
    text-align: center;
    border-bottom: 1px solid rgba(165,127,44,0.3);
    padding-bottom: 6px;
}}

.rutas-lista {{
    margin-bottom: 10px;
}}

.ruta-item {{
    background: rgba(30,30,30,0.8);
    border-radius: 8px;
    padding: 8px;
    margin-bottom: 8px;
    font-size: 11px;
    cursor: pointer;
    transition: all 0.2s;
}}

.ruta-item:hover {{
    background: rgba(165,127,44,0.2);
    border-left: 3px solid #4ecdc4;
}}

.ruta-origen-destino {{
    font-weight: bold;
    color: #4ecdc4;
    margin-bottom: 4px;
}}

.ruta-info {{
    color: #aaa;
    font-size: 10px;
    display: flex;
    justify-content: space-between;
}}

.ruta-boton {{
    width: 100%;
    background: linear-gradient(135deg, #1E5B4F, #A57F2C);
    color: white;
    border: none;
    padding: 8px;
    border-radius: 8px;
    font-size: 11px;
    font-weight: bold;
    cursor: pointer;
    margin-top: 5px;
    transition: all 0.2s;
}}

.ruta-boton:hover {{
    transform: translateY(-1px);
    box-shadow: 0 2px 8px rgba(165,127,44,0.4);
}}

.ruta-boton.limpiar {{
    background: rgba(100,100,100,0.6);
    margin-top: 5px;
}}

.ruta-boton.limpiar:hover {{
    background: rgba(150,150,150,0.8);
}}

.seleccion-activa {{
    position: fixed;
    top: 420px;
    left: 20px;
    z-index: 10001;
    background: rgba(165,127,44,0.2);
    backdrop-filter: blur(10px);
    padding: 5px 12px;
    border-radius: 20px;
    border: 1px solid #A57F2C;
    color: #A57F2C;
    font-size: 11px;
    font-weight: bold;
    pointer-events: none;
    display: none;
}}

.semaforo {{
    position: fixed;
    bottom: 20px;
    right: 20px;
    z-index: 10001;
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    padding: 12px 18px;
    border-radius: 12px;
    min-width: 240px;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    pointer-events: auto;
}}

.footer {{
    position: fixed;
    bottom: 20px;
    left: 320px;
    z-index: 10001;
    background: rgba(10,10,10,0.8);
    backdrop-filter: blur(10px);
    padding: 8px 12px;
    border-radius: 8px;
    font-size: 10px;
    color: #888;
    border: 1px solid rgba(165,127,44,0.2);
    pointer-events: auto;
}}

.deck-tooltip {{
    background: rgba(0,0,0,0.85) !important;
    backdrop-filter: blur(8px) !important;
    border: 1px solid rgba(165,127,44,0.4) !important;
    border-radius: 8px !important;
    box-shadow: 0 4px 15px rgba(0,0,0,0.5) !important;
    padding: 0 !important;
}}

.leaflet-popup-content-wrapper {{
    background: rgba(10,10,10,0.95);
    backdrop-filter: blur(10px);
    border: 1px solid rgba(165,127,44,0.3);
    border-radius: 12px;
    color: white;
    min-width: 200px;
}}

.leaflet-popup-tip {{
    background: rgba(10,10,10,0.95);
    border: 1px solid rgba(165,127,44,0.3);
}}

.leaflet-popup-content {{
    margin: 8px 12px;
    line-height: 1.4;
}}

.ruta-popup {{
    font-family: 'Segoe UI', Arial, sans-serif;
    text-align: center;
}}

.ruta-popup .distancia {{
    font-size: 16px;
    font-weight: bold;
    color: #4ecdc4;
    margin-bottom: 5px;
}}

.ruta-popup .tiempo {{
    font-size: 13px;
    color: #A57F2C;
}}

/* Estilo para puntos pequeños */
.custom-marker {{
    background: transparent;
    border: none;
}}

.custom-marker div {{
    width: 6px;
    height: 6px;
    border-radius: 50%;
    border: 1px solid rgba(0,0,0,0.3);
    box-shadow: 0 0 2px rgba(0,0,0,0.5);
    transition: all 0.2s ease;
}}

.custom-marker div:hover {{
    transform: scale(1.5);
    box-shadow: 0 0 4px rgba(165,127,44,0.8);
}}

/* Animación fosforescente para las rutas */
@keyframes phosphorescent {{
    0% {{
        stroke-opacity: 0.4;
        filter: drop-shadow(0 0 0px rgba(78, 205, 196, 0));
    }}
    50% {{
        stroke-opacity: 1;
        filter: drop-shadow(0 0 4px rgba(78, 205, 196, 0.8));
    }}
    100% {{
        stroke-opacity: 0.4;
        filter: drop-shadow(0 0 0px rgba(78, 205, 196, 0));
    }}
}}

.leaflet-routing-container {{
    display: none !important;
}}
</style>
</head>
<body>

<div id="map-3d"></div>
<div id="map-2d"></div>

<button class="btn-toggle" onclick="toggleModo()">Cambiar a Modo 2D (Rutas)</button>

<div class="header">
    <img src="https://imssbienestar.gob.mx/assets/img/imb_b.svg" 
         alt="IMSS Bienestar" 
         onerror="this.style.display='none'">
    Hospital Dashboard - Productividad por Unidad Medica
</div>

<div class="kpi-container">
    <div class="kpi-card">
        <div class="kpi-label">Unidades Activas</div>
        <div class="kpi-value">{total_unidades_activas:,}</div>
        <div class="kpi-sub">Cirugia: {unidades_cirugias} | Consulta: {unidades_consultas}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Avance General</div>
        <div class="kpi-value">{avg_avance_general:.1f}%</div>
        <div class="kpi-sub">Cirugia: {avg_avance_cirugias:.1f}% | Consulta: {avg_avance_consultas:.1f}%</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Volumen Realizado - Cirugía</div>
        <div class="kpi-value" style="color: #ff6b6b;">{volumen_cirugias:,.0f}</div>
        <div class="kpi-sub">Total de cirugías realizadas</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Volumen Realizado - Consulta</div>
        <div class="kpi-value" style="color: #4ecdc4;">{volumen_consultas:,.0f}</div>
        <div class="kpi-sub" style="font-size: 8px;">General: {consulta_general:,.0f} | Especialidad: {consulta_especialidad:,.0f}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Cobertura</div>
        <div class="kpi-value">{df_validos['entidad'].nunique()}</div>
        <div class="kpi-sub">Entidades Federativas</div>
    </div>
</div>

<div class="dashboard-btn">
    <button onclick="window.open('https://argontc.shinyapps.io/pptx/', '_blank')">
        Ver Dashboard de Reportes
    </button>
</div>

<div class="buscador">
    <div class="buscador-box">
        <div class="buscador-label">Buscador de Unidades</div>
        <input type="text" id="buscadorClues" class="buscador-input" placeholder="Buscar por CLUES o nombre...">
    </div>
    <div id="resultadosBusqueda" class="resultados"></div>
</div>

<div id="seleccionInfo" class="seleccion-activa"></div>

<div class="rutas-panel" id="rutasPanel">
    <h4>Trazado de Rutas (Modo 2D)</h4>
    <div id="unidadesSeleccionadas" class="rutas-lista">
        <div style="color: #888; font-size: 11px; text-align: center;">Selecciona unidades en el mapa 2D</div>
    </div>
    <button id="trazarRutaBtn" class="ruta-boton" onclick="trazarRuta2D()" disabled>Trazar Ruta</button>
    <button id="limpiarRutaBtn" class="ruta-boton limpiar" onclick="limpiarRuta2D()">Limpiar Ruta</button>
</div>

<div class="semaforo">
    <div style="color: white; font-size: 13px; font-weight: bold; margin-bottom: 10px; text-align: center; border-bottom: 1px solid rgba(165,127,44,0.3); padding-bottom: 5px;">
        Semaforo de Avance
    </div>
    <div style="font-size: 11px; color: #A57F2C; margin-bottom: 10px; text-align: center; background: rgba(0,0,0,0.5); padding: 5px; border-radius: 6px;">
        Año transcurrido: <strong>{pct_dia_esperado:.1f}%</strong>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['verde fuerte']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Verde Fuerte: Avance >= {verde_fuerte_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['verde claro']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Verde Claro: Avance >= {verde_claro_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['amarillo']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Amarillo: Avance >= {amarillo_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['rojo']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Rojo: Avance < {amarillo_umbral:.1f}%</span>
    </div>
</div>

<div class="footer">
    Datos al corte | Avance basado en meta esperada al dia de hoy
</div>

<script>
var unidadesData = {unidades_data_json};
var modo3D = true;
var selectedCluesList = [];
var currentRoute = null;
var map2D = null;
var routingControl = null;
var deckgl = null;

// Inicializar mapa 2D optimizado con puntos pequeños
function initMap2D() {{
    map2D = L.map('map-2d', {{
        attributionControl: false
    }}).setView([{lat_centro}, {lon_centro}], 6);
    
    L.tileLayer('https://{{s}}.basemaps.cartocdn.com/dark_all/{{z}}/{{x}}/{{y}}.png', {{
        subdomains: 'abcd',
        minZoom: 4,
        maxZoom: 18,
        attribution: ''
    }}).addTo(map2D);
    
    // Crear un grupo de capas para mejor rendimiento
    var markerLayer = L.layerGroup().addTo(map2D);
    
    // Agregar puntos pequeños
    unidadesData.forEach(function(unidad) {{
        var colorMap = {{
            'verde_fuerte': '#00FF00',
            'verde_claro': '#7FFF00', 
            'amarillo': '#FFD700',
            'rojo': '#FF4444'
        }};
        
        var markerColor = unidad.color_hex || colorMap[unidad.estado_semaforo] || '#888888';
        
        // Punto más pequeño (6px)
        var customIcon = L.divIcon({{
            className: 'custom-marker',
            html: '<div style="width: 6px; height: 6px; background-color: ' + markerColor + '; border-radius: 50%; border: 1px solid #A57F2C;"></div>',
            iconSize: [6, 6],
            iconAnchor: [3, 3],
            popupAnchor: [0, -3]
        }});
        
        var marker = L.marker([unidad.lat, unidad.lng], {{ 
            icon: customIcon,
            riseOnHover: true
        }}).addTo(markerLayer);
        
        // Popup con información básica
        marker.bindPopup(`
            <div style="font-family: 'Segoe UI', Arial, sans-serif; min-width: 180px;">
                <strong style="color: #A57F2C;">${{unidad.nombre_de_la_unidad}}</strong><br>
                <span style="font-size: 11px;">CLUES: ${{unidad.clues}}</span><br>
                <span style="font-size: 11px;">Avance: <span style="color:${{markerColor}}; font-weight:bold;">${{unidad.avance}}</span></span><br>
                <span style="font-size: 10px; color: #888;">${{unidad.estado_semaforo}}</span>
            </div>
        `);
        
        marker.on('click', function(e) {{
            L.DomEvent.stopPropagation(e);
            if (selectedCluesList.length < 2) {{
                if (!selectedCluesList.includes(unidad.clues)) {{
                    selectedCluesList.push(unidad.clues);
                    actualizarPanelSeleccion2D();
                    
                    var seleccionDiv = document.getElementById("seleccionInfo");
                    seleccionDiv.innerHTML = "Seleccionada: " + unidad.clues + " - " + unidad.nombre.substring(0, 25);
                    seleccionDiv.style.display = "block";
                    setTimeout(function() {{
                        seleccionDiv.style.display = "none";
                    }}, 2000);
                    
                    // Cambiar estilo del punto seleccionado temporalmente
                    var newIcon = L.divIcon({{
                        className: 'custom-marker',
                        html: '<div style="width: 10px; height: 10px; background-color: ' + markerColor + '; border-radius: 50%; border: 2px solid #4ecdc4; box-shadow: 0 0 6px #4ecdc4;"></div>',
                        iconSize: [10, 10],
                        iconAnchor: [5, 5],
                        popupAnchor: [0, -5]
                    }});
                    marker.setIcon(newIcon);
                    
                    setTimeout(function() {{
                        var originalIcon = L.divIcon({{
                            className: 'custom-marker',
                            html: '<div style="width: 6px; height: 6px; background-color: ' + markerColor + '; border-radius: 50%; border: 1px solid #A57F2C;"></div>',
                            iconSize: [6, 6],
                            iconAnchor: [3, 3],
                            popupAnchor: [0, -3]
                        }});
                        marker.setIcon(originalIcon);
                    }}, 1500);
                }}
            }} else {{
                alert("Solo se pueden seleccionar 2 unidades para trazar ruta. Usa 'Limpiar Ruta' para empezar de nuevo.");
            }}
        }});
    }});
}}

function actualizarPanelSeleccion2D() {{
    var panel = document.getElementById("unidadesSeleccionadas");
    var btnTrazar = document.getElementById("trazarRutaBtn");
    
    if (selectedCluesList.length === 0) {{
        panel.innerHTML = '<div style="color: #888; font-size: 11px; text-align: center;">Selecciona unidades en el mapa 2D</div>';
        btnTrazar.disabled = true;
        btnTrazar.textContent = "Trazar Ruta";
    }} else if (selectedCluesList.length === 1) {{
        var unidad1 = unidadesData.find(function(d) {{ return d.clues === selectedCluesList[0]; }});
        panel.innerHTML = 
            '<div class="ruta-item">' +
            '<div class="ruta-origen-destino">Origen: ' + unidad1.clues + '</div>' +
            '<div>' + unidad1.nombre.substring(0, 40) + '</div>' +
            '</div>' +
            '<div style="color: #A57F2C; font-size: 11px; text-align: center;">Selecciona una segunda unidad para la ruta</div>';
        btnTrazar.disabled = true;
        btnTrazar.textContent = "Selecciona destino";
    }} else if (selectedCluesList.length === 2) {{
        var unidadOrigen = unidadesData.find(function(d) {{ return d.clues === selectedCluesList[0]; }});
        var unidadDestino = unidadesData.find(function(d) {{ return d.clues === selectedCluesList[1]; }});
        panel.innerHTML = 
            '<div class="ruta-item">' +
            '<div class="ruta-origen-destino">Origen: ' + unidadOrigen.clues + '</div>' +
            '<div style="font-size: 10px;">' + unidadOrigen.nombre.substring(0, 35) + '</div>' +
            '</div>' +
            '<div class="ruta-item">' +
            '<div class="ruta-origen-destino">Destino: ' + unidadDestino.clues + '</div>' +
            '<div style="font-size: 10px;">' + unidadDestino.nombre.substring(0, 35) + '</div>' +
            '</div>';
        btnTrazar.disabled = false;
        btnTrazar.textContent = "Trazar Ruta";
    }}
}}

function trazarRuta2D() {{
    if (selectedCluesList.length !== 2) return;
    
    var origen = unidadesData.find(function(d) {{ return d.clues === selectedCluesList[0]; }});
    var destino = unidadesData.find(function(d) {{ return d.clues === selectedCluesList[1]; }});
    
    if (!origen || !destino) return;
    
    // Limpiar ruta anterior
    if (routingControl) {{
        map2D.removeControl(routingControl);
    }}
    
    // Crear nueva ruta con efecto fosforescente
    routingControl = L.Routing.control({{
        waypoints: [
            L.latLng(origen.lat, origen.lng),
            L.latLng(destino.lat, destino.lng)
        ],
        routeWhileDragging: false,
        showAlternatives: false,
        lineOptions: {{
            styles: [
                {{ 
                    color: '#4ecdc4', 
                    weight: 5, 
                    opacity: 0.8,
                    className: 'phosphorescent-line'
                }},
                {{
                    color: '#4ecdc4',
                    weight: 8,
                    opacity: 0.3,
                    className: 'phosphorescent-glow'
                }}
            ]
        }},
        createMarker: function() {{ return null; }},
        router: L.Routing.osrmv1({{
            serviceUrl: 'https://router.project-osrm.org/route/v1'
        }}),
        show: false
    }}).addTo(map2D);
    
    // Agregar animación fosforescente a la línea
    setTimeout(function() {{
        var svgPaths = document.querySelectorAll('.leaflet-overlay-pane svg path');
        svgPaths.forEach(function(path) {{
            path.style.animation = 'phosphorescent 1.5s ease-in-out infinite';
            path.style.filter = 'drop-shadow(0 0 4px rgba(78, 205, 196, 0.8))';
        }});
    }}, 100);
    
    routingControl.on('routesfound', function(e) {{
        var route = e.routes[0];
        var distance_km = (route.summary.totalDistance / 1000).toFixed(1);
        var duration_min = Math.round(route.summary.totalTime / 60);
        
        // Popup compacto solo con distancia y tiempo
        var popupContent = `
            <div class="ruta-popup">
                <div class="distancia">${{distance_km}} km</div>
                <div class="tiempo">${{duration_min}} minutos</div>
            </div>
        `;
        
        // Mostrar popup en el punto medio de la ruta
        var midLat = (origen.lat + destino.lat) / 2;
        var midLng = (origen.lng + destino.lng) / 2;
        L.popup()
            .setLatLng([midLat, midLng])
            .setContent(popupContent)
            .openOn(map2D);
        
        var panel = document.getElementById("unidadesSeleccionadas");
        var unidadOrigen = unidadesData.find(function(d) {{ return d.clues === selectedCluesList[0]; }});
        var unidadDestino = unidadesData.find(function(d) {{ return d.clues === selectedCluesList[1]; }});
        panel.innerHTML = 
            '<div class="ruta-item">' +
            '<div class="ruta-origen-destino">Origen: ' + unidadOrigen.clues + '</div>' +
            '<div style="font-size: 10px;">' + unidadOrigen.nombre.substring(0, 35) + '</div>' +
            '</div>' +
            '<div class="ruta-item">' +
            '<div class="ruta-origen-destino">Destino: ' + unidadDestino.clues + '</div>' +
            '<div style="font-size: 10px;">' + unidadDestino.nombre.substring(0, 35) + '</div>' +
            '<div class="ruta-info">' +
            '<span>Distancia: ' + distance_km + ' km</span>' +
            '<span>Duracion: ' + duration_min + ' min</span>' +
            '</div>' +
            '</div>';
        
        // Ajustar vista para mostrar toda la ruta
        var bounds = L.latLngBounds([origen.lat, origen.lng], [destino.lat, destino.lng]);
        map2D.fitBounds(bounds, {{ padding: [50, 50] }});
        
        // Re-aplicar animación después de ajustar vista
        setTimeout(function() {{
            var svgPaths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            svgPaths.forEach(function(path) {{
                path.style.animation = 'phosphorescent 1.5s ease-in-out infinite';
                path.style.filter = 'drop-shadow(0 0 4px rgba(78, 205, 196, 0.8))';
            }});
        }}, 200);
    }});
}}

function limpiarRuta2D() {{
    selectedCluesList = [];
    if (routingControl) {{
        map2D.removeControl(routingControl);
        routingControl = null;
    }}
    actualizarPanelSeleccion2D();
    map2D.closePopup();
}}

function toggleModo() {{
    var map3dDiv = document.getElementById("map-3d");
    var map2dDiv = document.getElementById("map-2d");
    var rutasPanel = document.getElementById("rutasPanel");
    var btn = document.querySelector(".btn-toggle");
    
    modo3D = !modo3D;
    
    if (modo3D) {{
        map3dDiv.style.display = "block";
        map2dDiv.style.display = "none";
        rutasPanel.style.display = "none";
        btn.textContent = "Cambiar a Modo 2D (Rutas)";
        
        selectedCluesList = [];
        if (routingControl) {{
            routingControl = null;
        }}
    }} else {{
        map3dDiv.style.display = "none";
        map2dDiv.style.display = "block";
        rutasPanel.style.display = "block";
        btn.textContent = "Cambiar a Modo 3D";
        
        if (!map2D) {{
            initMap2D();
        }} else {{
            map2D.invalidateSize();
        }}
    }}
}}

// Configurar mapa 3D con deck.gl
function initMap3D() {{
    var currentViewState = {{
        longitude: {lon_centro},
        latitude: {lat_centro},
        zoom: 5.5,
        pitch: 55,
        bearing: 0
    }};
    
    deckgl = new deck.DeckGL({{
        container: "map-3d",
        mapStyle: "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
        initialViewState: currentViewState,
        controller: true,
        getTooltip: function(info) {{
            if (!info.object) return null;
            var d = info.object;
            return {{
                html: '<div style="font-family: Segoe UI, Arial, sans-serif; min-width:260px; padding: 10px;">' +
                    '<div style="font-weight:bold;font-size:13px;margin-bottom:6px;border-bottom: 1px solid rgba(165,127,44,0.3);padding-bottom: 4px;color:#A57F2C;">' +
                    d.nombre_de_la_unidad +
                    '</div>' +
                    '<div style="display: grid; grid-template-columns: 85px 1fr; gap: 4px; font-size: 11px;">' +
                    '<span style="color:#aaa;">CLUES:</span>' +
                    '<span style="font-weight:500;">' + d.clues + '</span>' +
                    '<span style="color:#aaa;">Entidad:</span>' +
                    '<span>' + d.entidad + '</span>' +
                    '<span style="color:#aaa;">Nivel:</span>' +
                    '<span>' + d.nivel + '</span>' +
                    '<span style="color:#aaa;">Indicador:</span>' +
                    '<span>' + d.tipo + '</span>' +
                    '<span style="color:#aaa;">Realizado:</span>' +
                    '<span style="font-weight:500;">' + d.realizado + '</span>' +
                    '<span style="color:#aaa;">Meta Anual:</span>' +
                    '<span>' + d.meta_anual + '</span>' +
                    '<span style="color:#aaa;">Meta Esperada:</span>' +
                    '<span>' + d.meta_esperada + '</span>' +
                    '<span style="color:#aaa;">Avance Real:</span>' +
                    '<span style="color:' + d.color_hex + '; font-weight:bold;">' + d.avance + '</span>' +
                    '<span style="color:#aaa;">Avance Esperado:</span>' +
                    '<span>' + d.pct_dia_fmt + '</span>' +
                    '</div>' +
                    '<div style="margin-top: 6px; padding-top: 4px; border-top: 1px solid rgba(165,127,44,0.2); font-size: 10px; color: #A57F2C;">' +
                    d.estado_semaforo +
                    '</div></div>'
            }};
        }},
        layers: [
            new deck.ColumnLayer({{
                id: "columnas-avance",
                data: unidadesData,
                diskResolution: 12,
                radius: 400,
                extruded: true,
                pickable: true,
                elevationScale: 0.5,
                getPosition: function(d) {{ return [d.lng, d.lat]; }},
                getElevation: function(d) {{ return d.altura; }},
                getFillColor: function(d) {{
                    return [d.r, d.g, d.b, 200];
                }}
            }})
        ]
    }});
}}

// Inicializar
initMap3D();

// Buscador
var buscadorInput = document.getElementById("buscadorClues");
var resultadosDiv = document.getElementById("resultadosBusqueda");

function buscarUnidades(texto) {{
    if (texto.length < 2) {{
        resultadosDiv.style.display = "none";
        return [];
    }}
    texto = texto.toLowerCase();
    return unidadesData.filter(function(u) {{
        return (
            u.clues.toLowerCase().includes(texto) ||
            u.nombre.toLowerCase().includes(texto)
        );
    }}).slice(0, 10);
}}

function centrarEnUnidad(unidad) {{
    if (modo3D && deckgl) {{
        deckgl.setProps({{
            initialViewState: {{
                longitude: unidad.lng,
                latitude: unidad.lat,
                zoom: 14,
                pitch: 60,
                bearing: 0,
                transitionDuration: 1500,
                transitionInterpolator: new deck.FlyToInterpolator({{ speed: 1.2 }})
            }}
        }});
    }} else if (map2D) {{
        map2D.setView([unidad.lat, unidad.lng], 14);
    }}
    
    buscadorInput.value = unidad.clues;
    resultadosDiv.style.display = "none";
}}

function mostrarResultados(resultados) {{
    resultadosDiv.innerHTML = "";
    if (resultados.length === 0) {{
        resultadosDiv.style.display = "none";
        return;
    }}
    resultados.forEach(function(u) {{
        var item = document.createElement("div");
        item.className = "resultado-item";
        item.innerHTML = 
            '<div style="font-weight:bold;color:#A57F2C;font-size:13px;">' + u.clues + '</div>' +
            '<div style="font-size:11px;color:white;margin-top:3px;">' + u.nombre.substring(0,50) + '</div>' +
            '<div style="font-size:10px;color:#A57F2C;margin-top:5px;">Avance: ' + u.avance + '</div>';
        item.onclick = function() {{ centrarEnUnidad(u); }};
        resultadosDiv.appendChild(item);
    }});
    resultadosDiv.style.display = "block";
}}

buscadorInput.addEventListener("input", function(e) {{
    var resultados = buscarUnidades(e.target.value);
    mostrarResultados(resultados);
}});

document.addEventListener("click", function(e) {{
    if (buscadorInput && resultadosDiv) {{
        if (!buscadorInput.contains(e.target) && !resultadosDiv.contains(e.target)) {{
            resultadosDiv.style.display = "none";
        }}
    }}
}});
</script>
</body>
</html>
"""


webbrowser.open(archivo_salida.resolve().as_uri())

True

## correr

## intento de mapa 3d

In [263]:
archivo_salida = Path("index.html")

with open(archivo_salida, "w", encoding="utf-8") as f:
    f.write(html_dual)

webbrowser.open(archivo_salida.resolve().as_uri())

True

In [264]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:
df_3d

,clues_imb,entidad,nombre_de_la_unidad,nivel_atencion,cirugias,meta_cirugia_anual,consultas,meta_total,latitud,longitud,...,realizado_fmt,meta_anual_fmt,meta_esperada_fmt,tipo_indicador,pct_general_num,altura,color_semaforo_hex,r,g,b
0,VZIMB007522,VERACRUZ,COL. CARRILES,PRIMER NIVEL,0.0,0,3795.0,15937,18.888400,-96.940200,...,"3,795","15,937","6,724",consultas,0.564,11780.0,#0D5D2A,13,93,42
1,TLIMB002182,TLAXCALA,GUADALUPE IXCOTLA (TECTEYOC),PRIMER NIVEL,0.0,0,2007.0,6124,19.325992,-98.185950,...,"2,007","6,124","2,583",consultas,0.777,16040.0,#0D5D2A,13,93,42
2,VZIMB008222,VERACRUZ,CENTRO COMUNITARIO DE SALUD MENTAL Y ADICCIONE...,PRIMER NIVEL,0.0,0,194.0,1072,19.103010,-96.103300,...,194,"1,072",452,consultas,0.429,9080.0,#0D5D2A,13,93,42
3,CMIMB000844,COLIMA,HOSPITAL GENERAL TECOMÁN DR. JOSÉ F. RIVAS GUZMÁN,SEGUNDO NIVEL,875.0,3491,875.0,26288,18.906326,-103.884075,...,875,"3,491","1,472",cirugias,0.594,12380.0,#0D5D2A,13,93,42
4,TSIMB001424,TAMAULIPAS,CENTRO DE SALUD HIJOS DE EJIDATARIOS,PRIMER NIVEL,0.0,0,455.0,1827,25.971298,-98.120751,...,455,"1,827",770,consultas,0.590,12300.0,#0D5D2A,13,93,42
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10572,QRIMB001290,QUINTANA ROO,CENTRO DE SALUD RURAL EL CEDRAL,PRIMER NIVEL,0.0,0,0.0,2109,20.950242,-87.549986,...,0,"2,109",889,consultas,0.000,500.0,#D41111,212,17,17
10573,SLIMB002406,SINALOA,UNIDAD MÓVIL DE LA MUJER,PRIMER NIVEL,0.0,0,0.0,1810,24.810092,-107.383914,...,0,"1,810",763,consultas,0.000,500.0,#D41111,212,17,17
10575,TSIMB003676,TAMAULIPAS,CENTRO DE SALUD PALO SOLO,PRIMER NIVEL,0.0,0,0.0,2920,25.085069,-97.918022,...,0,"2,920","1,232",consultas,0.000,500.0,#D41111,212,17,17
10576,VZIMB004990,VERACRUZ,URSULO GALVAN (LAS CHARCAS),PRIMER NIVEL,0.0,0,0.0,2911,18.548800,-96.056600,...,0,"2,911","1,228",consultas,0.000,500.0,#D41111,212,17,17


## prueba

In [ ]:


df_3d = df_validos.copy()

# Calcular el porcentaje del año transcurrido basado en el último miércoles
hoy = datetime.now()
dias_desde_miercoles = (hoy.weekday() - 2) % 7
if dias_desde_miercoles == 0:
    dias_desde_miercoles = 7
else:
    dias_desde_miercoles += 7

ultimo_miercoles = hoy - timedelta(days=dias_desde_miercoles)
dias_transcurridos = ultimo_miercoles.timetuple().tm_yday
dias_del_anio = 365 + int(ultimo_miercoles.year % 4 == 0 and (ultimo_miercoles.year % 100 != 0 or ultimo_miercoles.year % 400 == 0))
pct_tiempo_decimal = dias_transcurridos / dias_del_anio

pct_dia_esperado = pct_tiempo_decimal * 100

# Actualizar variable global
PCT_ANIO_PORCENTAJE = pct_dia_esperado

# Procesar datos numéricos
if df_3d["pct_general"].dtype == "object":
    df_3d["pct_general_num"] = pd.to_numeric(df_3d["pct_general"].str.rstrip('%'), errors='coerce') / 100
else:
    if df_3d["pct_general"].max() > 1:
        df_3d["pct_general_num"] = df_3d["pct_general"] / 100
    else:
        df_3d["pct_general_num"] = df_3d["pct_general"]

df_3d["pct_general_num"] = df_3d["pct_general_num"].fillna(0).clip(0, 1)

# Altura de columnas
df_3d["altura"] = 500 + (df_3d["pct_general_num"] * 20000)


In [ ]:
df_3d

,clues_imb,entidad,nombre_de_la_unidad,nivel_atencion,cirugias,meta_cirugia_anual,consultas,meta_total,latitud,longitud,...,macro_key,color_hex,pct_general_fmt,estado_semaforo,realizado_fmt,meta_anual_fmt,meta_esperada_fmt,tipo_indicador,pct_general_num,altura
0,VZIMB008193,VERACRUZ,CENTRO COMUNITARIO DE SALUD MENTAL Y ADICCIONE...,PRIMER NIVEL,0.0,0,472.0,1072,18.873279,-97.109430,...,consultas,#0D5D2A,44.0%,Verde Fuerte (Avance >= 40.1%),472,"1,072",452,consultas,0.440,9300.0
1,NTIMB002255,NAYARIT,CENTRO COMUNITARIO DE SALUD MENTAL Y ADICCIONE...,PRIMER NIVEL,0.0,0,607.0,1069,20.746366,-105.298885,...,consultas,#0D5D2A,56.8%,Verde Fuerte (Avance >= 40.1%),607,"1,069",451,consultas,0.568,11860.0
2,MCIMB003936,MEXICO,REFORMA,PRIMER NIVEL,0.0,0,8002.0,29399,19.374249,-98.975576,...,consultas,#F1D54A,27.2%,Amarillo (Avance >= 25.3%),"8,002","29,399","12,403",consultas,0.272,5940.0
3,MCIMB004011,MEXICO,COL.BENITO JUÁREZ II EL VERGELITO,PRIMER NIVEL,0.0,0,13633.0,48377,19.410176,-99.006231,...,consultas,#F1D54A,28.2%,Amarillo (Avance >= 25.3%),"13,633","48,377","20,411",consultas,0.282,6140.0
4,VZIMB008263,VERACRUZ,UNIDAD DE ESPECIALIDADES MÉDICAS DE SALUD MENT...,PRIMER NIVEL,0.0,0,3157.0,8258,18.137800,-94.435300,...,consultas,#88A91E,38.2%,Verde Claro (Avance >= 33.8%),"3,157","8,258","3,484",consultas,0.382,8140.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10572,QRIMB001290,QUINTANA ROO,CENTRO DE SALUD RURAL EL CEDRAL,PRIMER NIVEL,0.0,0,0.0,2109,20.950242,-87.549986,...,consultas,#D41111,0.0%,Rojo (Avance < 25.3%),0,"2,109",889,consultas,0.000,500.0
10573,SLIMB002406,SINALOA,UNIDAD MÓVIL DE LA MUJER,PRIMER NIVEL,0.0,0,0.0,1810,24.810092,-107.383914,...,consultas,#D41111,0.0%,Rojo (Avance < 25.3%),0,"1,810",763,consultas,0.000,500.0
10575,TSIMB003676,TAMAULIPAS,CENTRO DE SALUD PALO SOLO,PRIMER NIVEL,0.0,0,0.0,2920,25.085069,-97.918022,...,consultas,#D41111,0.0%,Rojo (Avance < 25.3%),0,"2,920","1,232",consultas,0.000,500.0
10576,VZIMB004990,VERACRUZ,URSULO GALVAN (LAS CHARCAS),PRIMER NIVEL,0.0,0,0.0,2911,18.548800,-96.056600,...,consultas,#D41111,0.0%,Rojo (Avance < 25.3%),0,"2,911","1,228",consultas,0.000,500.0


In [ ]:

# ========== SEMAFORO ==========
def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

df_3d["color_semaforo_hex"] = df_3d["pct_general"].apply(obtener_color_semaforo)
df_3d[["r", "g", "b"]] = df_3d["color_semaforo_hex"].apply(lambda x: pd.Series(hex_to_rgb(x)))

# Umbrales basados en el porcentaje del año transcurrido
verde_fuerte_umbral = PCT_ANIO_PORCENTAJE * 0.95
verde_claro_umbral = PCT_ANIO_PORCENTAJE * 0.80
amarillo_umbral = PCT_ANIO_PORCENTAJE * 0.60

# ========== VOLUMENES CON DESGLOSE ==========
volumen_cirugias = pd.to_numeric(conteo_consultas.loc[0, "cirugia"], errors="coerce")
volumen_consultas = pd.to_numeric(conteo_consultas.loc[0, "consulta"], errors="coerce")
consulta_general = pd.to_numeric(conteo_consu.loc[0, "general"], errors="coerce")
consulta_especialidad = pd.to_numeric(conteo_consu.loc[0, "especialidad"], errors="coerce")

volumen_cirugias = 0 if pd.isna(volumen_cirugias) else volumen_cirugias
volumen_consultas = 0 if pd.isna(volumen_consultas) else volumen_consultas
consulta_general = 0 if pd.isna(consulta_general) else consulta_general
consulta_especialidad = 0 if pd.isna(consulta_especialidad) else consulta_especialidad

# ========== AVANCE GENERAL - CORRECTO (usando sumas totales) ==========
# Calcular usando tus dataframes originales
df_cirugias_total = df_validos[df_validos["macro_key"] == "cirugias"].copy()
df_consultas_total = df_validos[df_validos["macro_key"] == "consultas"].copy()

if len(df_cirugias_total) > 0:
    total_cirugias = df_cirugias_total["cirugias"].sum()
    total_cirugias = total_cirugias if not pd.isna(total_cirugias) else 0
    meta_cirugias_total = df_cirugias_total["meta_cirugia_anual"].sum()
    meta_cirugias_total = meta_cirugias_total if not pd.isna(meta_cirugias_total) else 0
else:
    total_cirugias = 0
    meta_cirugias_total = 0

if len(df_consultas_total) > 0:
    total_consultas = df_consultas_total["consultas"].sum()
    total_consultas = total_consultas if not pd.isna(total_consultas) else 0
    meta_consultas_total = df_consultas_total["meta_total"].sum()
    meta_consultas_total = meta_consultas_total if not pd.isna(meta_consultas_total) else 0
else:
    total_consultas = 0
    meta_consultas_total = 0

total_realizado = total_cirugias + total_consultas
meta_anual_total = meta_cirugias_total + meta_consultas_total

if meta_anual_total > 0:
    avg_avance_general = total_realizado / meta_anual_total * 100
    avg_avance_general = avg_avance_general if not pd.isna(avg_avance_general) else 0
else:
    avg_avance_general = 0

if meta_cirugias_total > 0:
    avg_avance_cirugias = total_cirugias / meta_cirugias_total * 100
    avg_avance_cirugias = avg_avance_cirugias if not pd.isna(avg_avance_cirugias) else 0
else:
    avg_avance_cirugias = 0

if meta_consultas_total > 0:
    avg_avance_consultas = total_consultas / meta_consultas_total * 100
    avg_avance_consultas = avg_avance_consultas if not pd.isna(avg_avance_consultas) else 0
else:
    avg_avance_consultas = 0

# Unidades activas (con avance > 0)
df_sin_ceros = df_validos[df_validos["pct_general"] > 0].copy()
unidades_cirugias_activas = len(df_sin_ceros[df_sin_ceros["macro_key"] == "cirugias"])
unidades_consultas_activas = len(df_sin_ceros[df_sin_ceros["macro_key"] == "consultas"])
total_unidades_activas = len(df_validos)
entidades_unicas = df_validos['entidad'].nunique() if 'entidad' in df_validos.columns else 0

# Datos para buscador
unidades_data = []
for _, row in df_3d.iterrows():
    unidades_data.append({
        "clues": row['clues_imb'],
        "nombre": row['nombre_de_la_unidad'],
        "lat": row['latitud'],
        "lng": row['longitud'],
        "avance": row['pct_general_fmt'],
        "avance_num": row['pct_general_num'] * 100,
        "entidad": row['entidad'],
        "nivel": row['nivel_atencion'],
        "tipo": row['tipo_indicador'],
        "realizado": row['realizado_fmt'],
        "meta_anual": row['meta_anual_fmt'],
        "meta_esperada": row['meta_esperada_fmt'],
        "pct_dia_fmt": row['pct_dia_fmt'],
        "pct_dia_num": row['pct_tiempo'] if 'pct_tiempo' in row else 0,
        "color_hex": row['color_semaforo_hex'],
        "estado_semaforo": row['estado_semaforo'],
        "nombre_de_la_unidad": row['nombre_de_la_unidad'],
        "altura": row['altura'],
        "r": row['r'],
        "g": row['g'],
        "b": row['b']
    })
unidades_data_json = json.dumps(unidades_data, ensure_ascii=False)

# Centro del mapa
lat_centro = df_3d["latitud"].mean()
lon_centro = df_3d["longitud"].mean()



# SVG para loader
svg_path = r"C:\Users\jose.valdez\Downloads\IMSS_BIENESTAR.svg"
try:
    with open(svg_path, "r", encoding="utf-8") as f:
        svg_content = f.read()
    svg_base64 = base64.b64encode(svg_content.encode('utf-8')).decode('utf-8')
    svg_data_uri = f"data:image/svg+xml;base64,{svg_base64}"
except Exception as e:
    print(f"No se pudo cargar el SVG: {e}")
    svg_data_uri = ""


In [269]:

# ============================================================
# HTML COMPLETO CON 2D Y 3D 
# ============================================================

html_completo = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<title>Hospital Dashboard - Modo 3D y 2D para Rutas</title>

<script src="https://unpkg.com/deck.gl@latest/dist.min.js"></script>
<script src="https://unpkg.com/maplibre-gl@latest/dist/maplibre-gl.js"></script>
<link href="https://unpkg.com/maplibre-gl@latest/dist/maplibre-gl.css" rel="stylesheet"/>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://unpkg.com/leaflet-routing-machine@latest/dist/leaflet-routing-machine.js"></script>
<link rel="stylesheet" href="https://unpkg.com/leaflet-routing-machine@latest/dist/leaflet-routing-machine.css" />

<style>
html, body {{
    margin: 0;
    width: 100%;
    height: 100%;
    font-family: 'Segoe UI', Arial, sans-serif;
    background: #0a0a0a;
}}

#map-3d {{
    width: 100%;
    height: 100%;
    background: #0d0d0d;
    display: block;
}}

#map-2d {{
    width: 100%;
    height: 100%;
    background: #0d0d0d;
    display: none;
}}

.leaflet-control-attribution {{
    display: none !important;
}}

/* LOADER */
.loader-overlay {{
    position: fixed;
    top: 0;
    left: 0;
    width: 100%;
    height: 100%;
    background: rgba(0,0,0,0.9);
    backdrop-filter: blur(8px);
    z-index: 20000;
    display: flex;
    align-items: center;
    justify-content: center;
    flex-direction: column;
    opacity: 0;
    visibility: hidden;
    transition: opacity 0.2s ease, visibility 0s linear 0.3s;
    pointer-events: none;
}}

.loader-overlay.active {{
    opacity: 1;
    visibility: visible;
    transition: opacity 0.2s ease, visibility 0s linear 0s;
    pointer-events: all;
}}

.loader-text {{
    color: #A57F2C;
    margin-top: 30px;
    font-size: 14px;
    font-weight: bold;
    letter-spacing: 2px;
    text-transform: uppercase;
    background: rgba(0,0,0,0.6);
    padding: 8px 20px;
    border-radius: 30px;
    border: 1px solid rgba(165,127,44,0.3);
}}

.loader-svg {{
    width: 450px;
    height: 300px;
    animation: spin 5s linear infinite;
    filter: drop-shadow(0 0 15px rgba(165,127,44,0.6));
}}

@keyframes spin {{
    0% {{ transform: rotate(0deg); }}
    100% {{ transform: rotate(360deg); }}
}}

.header {{
    position: fixed;
    top: 0;
    left: 0;
    right: 0;
    z-index: 10000;
    background: #1E5B4F;
    color: white;
    padding: 12px 25px;
    font-size: 18px;
    font-weight: bold;
    text-align: center;
    box-shadow: 0 2px 10px rgba(0,0,0,0.5);
    pointer-events: none;
    border-bottom: 1px solid rgba(165,127,44,0.3);
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 15px;
}}

.header img {{
    height: 40px;
    width: auto;
    filter: brightness(0) invert(1);
}}

.btn-toggle {{
    position: fixed;
    top: 80px;
    right: 20px;
    z-index: 10001;
    background: linear-gradient(135deg, #A57F2C, #1E5B4F);
    color: white;
    border: none;
    padding: 12px 24px;
    border-radius: 30px;
    font-size: 14px;
    font-weight: bold;
    cursor: pointer;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-shadow: 0 4px 15px rgba(0,0,0,0.3);
    transition: all 0.3s;
    pointer-events: auto;
}}

.btn-toggle:hover {{
    transform: translateY(-2px);
    box-shadow: 0 6px 20px rgba(165,127,44,0.4);
}}

.kpi-container {{
    position: fixed;
    top: 70px;
    left: 0;
    right: 0;
    z-index: 10000;
    display: grid;
    grid-template-columns: repeat(5, 1fr);
    gap: 12px;
    padding: 12px 20px;
    pointer-events: none;
}}

.kpi-card {{
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    padding: 10px 15px;
    border-radius: 10px;
    border: 1px solid rgba(165,127,44,0.3);
    pointer-events: auto;
    box-shadow: 0 2px 5px rgba(0,0,0,0.3);
}}

.kpi-label {{
    font-size: 10px;
    color: #A57F2C;
    text-transform: uppercase;
    letter-spacing: 1px;
}}

.kpi-value {{
    font-size: 24px;
    font-weight: bold;
    color: white;
}}

.kpi-sub {{
    font-size: 9px;
    color: #888;
}}

.dashboard-btn {{
    position: fixed;
    top: 380px;
    left: 20px;
    z-index: 10001;
}}

.dashboard-btn button {{
    background: linear-gradient(135deg, #A57F2C, #1E5B4F);
    color: white;
    border: none;
    padding: 8px 18px;
    border-radius: 25px;
    font-size: 12px;
    font-weight: bold;
    cursor: pointer;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-shadow: 0 2px 5px rgba(0,0,0,0.3);
    transition: all 0.3s;
    pointer-events: auto;
}}

.dashboard-btn button:hover {{
    transform: translateY(-2px);
    box-shadow: 0 4px 10px rgba(165,127,44,0.3);
}}

.buscador {{
    position: fixed;
    top: 200px;
    left: 20px;
    z-index: 10001;
    width: 320px;
}}

.buscador-box {{
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    border-radius: 12px;
    padding: 12px;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
}}

.buscador-label {{
    color: #A57F2C;
    font-size: 10px;
    text-transform: uppercase;
    margin-bottom: 8px;
    letter-spacing: 1px;
}}

.buscador-input {{
    width: 100%;
    padding: 10px 15px;
    background: rgba(20,20,20,0.9);
    border: 1px solid rgba(165,127,44,0.3);
    border-radius: 8px;
    color: white;
    font-size: 13px;
    outline: none;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-sizing: border-box;
}}

.buscador-input:focus {{
    border-color: #A57F2C;
    box-shadow: 0 0 5px rgba(165,127,44,0.5);
}}

.resultados {{
    background: rgba(10,10,10,0.95);
    border-radius: 12px;
    margin-top: 8px;
    max-height: 350px;
    overflow-y: auto;
    display: none;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
}}

.resultado-item {{
    padding: 12px 15px;
    cursor: pointer;
    border-bottom: 1px solid rgba(165,127,44,0.1);
    font-family: 'Segoe UI', Arial, sans-serif;
    transition: all 0.2s;
}}

.resultado-item:hover {{
    background: rgba(165,127,44,0.15);
    border-left: 3px solid #A57F2C;
}}

.rutas-panel {{
    position: fixed;
    bottom: 20px;
    left: 20px;
    z-index: 10001;
    background: rgba(10,10,10,0.95);
    backdrop-filter: blur(10px);
    border-radius: 12px;
    min-width: 280px;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    pointer-events: auto;
    padding: 12px;
    display: none;
}}

.rutas-panel h4 {{
    margin: 0 0 10px 0;
    color: #A57F2C;
    font-size: 13px;
    text-align: center;
    border-bottom: 1px solid rgba(165,127,44,0.3);
    padding-bottom: 6px;
}}

.rutas-lista {{
    margin-bottom: 10px;
}}

.ruta-item {{
    background: rgba(30,30,30,0.8);
    border-radius: 8px;
    padding: 8px;
    margin-bottom: 8px;
    font-size: 11px;
    cursor: pointer;
    transition: all 0.2s;
}}

.ruta-item:hover {{
    background: rgba(165,127,44,0.2);
    border-left: 3px solid #4ecdc4;
}}

.ruta-origen-destino {{
    font-weight: bold;
    color: #4ecdc4;
    margin-bottom: 4px;
}}

.ruta-info {{
    color: #aaa;
    font-size: 10px;
    display: flex;
    justify-content: space-between;
}}

.ruta-boton {{
    width: 100%;
    background: linear-gradient(135deg, #1E5B4F, #A57F2C);
    color: white;
    border: none;
    padding: 8px;
    border-radius: 8px;
    font-size: 11px;
    font-weight: bold;
    cursor: pointer;
    margin-top: 5px;
    transition: all 0.2s;
}}

.ruta-boton:hover {{
    transform: translateY(-1px);
    box-shadow: 0 2px 8px rgba(165,127,44,0.4);
}}

.ruta-boton.limpiar {{
    background: rgba(100,100,100,0.6);
    margin-top: 5px;
}}

.ruta-boton.limpiar:hover {{
    background: rgba(150,150,150,0.8);
}}

.seleccion-activa {{
    position: fixed;
    top: 420px;
    left: 20px;
    z-index: 10001;
    background: rgba(165,127,44,0.2);
    backdrop-filter: blur(10px);
    padding: 5px 12px;
    border-radius: 20px;
    border: 1px solid #A57F2C;
    color: #A57F2C;
    font-size: 11px;
    font-weight: bold;
    pointer-events: none;
    display: none;
}}

.semaforo {{
    position: fixed;
    bottom: 20px;
    right: 20px;
    z-index: 10001;
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    padding: 12px 18px;
    border-radius: 12px;
    min-width: 240px;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    pointer-events: auto;
}}

.footer {{
    position: fixed;
    bottom: 20px;
    left: 320px;
    z-index: 10001;
    background: rgba(10,10,10,0.8);
    backdrop-filter: blur(10px);
    padding: 8px 12px;
    border-radius: 8px;
    font-size: 10px;
    color: #888;
    border: 1px solid rgba(165,127,44,0.2);
    pointer-events: auto;
}}

.deck-tooltip {{
    background: rgba(0,0,0,0.85) !important;
    backdrop-filter: blur(8px) !important;
    border: 1px solid rgba(165,127,44,0.4) !important;
    border-radius: 8px !important;
    box-shadow: 0 4px 15px rgba(0,0,0,0.5) !important;
    padding: 0 !important;
}}

.leaflet-popup-content-wrapper {{
    background: rgba(10,10,10,0.95);
    backdrop-filter: blur(10px);
    border: 1px solid rgba(165,127,44,0.3);
    border-radius: 12px;
    color: white;
    min-width: 200px;
}}

.leaflet-popup-tip {{
    background: rgba(10,10,10,0.95);
    border: 1px solid rgba(165,127,44,0.3);
}}

.leaflet-popup-content {{
    margin: 8px 12px;
    line-height: 1.4;
}}

.ruta-popup {{
    font-family: 'Segoe UI', Arial, sans-serif;
    text-align: center;
}}

.ruta-popup .distancia {{
    font-size: 16px;
    font-weight: bold;
    color: #4ecdc4;
    margin-bottom: 5px;
}}

.ruta-popup .tiempo {{
    font-size: 13px;
    color: #A57F2C;
}}

.custom-marker {{
    background: transparent;
    border: none;
}}

.custom-marker div {{
    width: 6px;
    height: 6px;
    border-radius: 50%;
    border: 1px solid rgba(0,0,0,0.3);
    box-shadow: 0 0 2px rgba(0,0,0,0.5);
    transition: all 0.2s ease;
}}

.custom-marker div:hover {{
    transform: scale(1.5);
    box-shadow: 0 0 4px rgba(165,127,44,0.8);
}}

.selected-marker div {{
    width: 16px !important;
    height: 16px !important;
    border: 3px solid #4ecdc4 !important;
    box-shadow: 0 0 15px rgba(78, 205, 196, 0.8) !important;
    animation: pulse 1.5s ease-in-out infinite !important;
}}

@keyframes pulse {{
    0% {{ transform: scale(1); opacity: 1; }}
    50% {{ transform: scale(1.3); opacity: 0.7; }}
    100% {{ transform: scale(1); opacity: 1; }}
}}

.leaflet-routing-container {{
    display: none !important;
}}
</style>
</head>
<body>

<div id="map-3d"></div>
<div id="map-2d"></div>

<div id="loaderOverlay" class="loader-overlay">
    <img class="loader-svg" src="{svg_data_uri}" alt="Cargando...">
    <div class="loader-text">Cargando mapa...</div>
</div>

<button class="btn-toggle" onclick="toggleModo()">Cambiar a Modo 2D (Rutas)</button>

<div class="header">
    <img src="https://imssbienestar.gob.mx/assets/img/imb_b.svg" 
         alt="IMSS Bienestar" 
         onerror="this.style.display='none'">
    Hospital Dashboard - Productividad por Unidad Medica
</div>

<div class="kpi-container">
    <div class="kpi-card">
        <div class="kpi-label">Unidades Activas</div>
        <div class="kpi-value">{total_unidades_activas:,}</div>
        <div class="kpi-sub">Cirugia: {unidades_cirugias_activas} | Consulta: {unidades_consultas_activas}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Avance General</div>
        <div class="kpi-value">{avg_avance_general:.1f}%</div>
        <div class="kpi-sub">Cirugia: {avg_avance_cirugias:.1f}% | Consulta: {avg_avance_consultas:.1f}%</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Volumen Realizado - Cirugia</div>
        <div class="kpi-value" style="color: #ffffff;">{volumen_cirugias:,.0f}</div>
        <div class="kpi-sub">Total de cirugias realizadas</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Volumen Realizado - Consulta</div>
        <div class="kpi-value" style="color: #4ecdc4;">{volumen_consultas:,.0f}</div>
        <div class="kpi-sub" style="font-size: 8px;">General: {consulta_general:,.0f} | Especialidad: {consulta_especialidad:,.0f}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Cobertura</div>
        <div class="kpi-value">{entidades_unicas}</div>
        <div class="kpi-sub">Entidades Federativas</div>
    </div>
</div>

<div class="dashboard-btn">
    <button onclick="window.open('https://argontc.shinyapps.io/pptx/', '_blank')">
        Ver Dashboard de Reportes
    </button>
</div>

<div class="buscador">
    <div class="buscador-box">
        <div class="buscador-label">Buscador de Unidades</div>
        <input type="text" id="buscadorClues" class="buscador-input" placeholder="Buscar por CLUES o nombre...">
    </div>
    <div id="resultadosBusqueda" class="resultados"></div>
</div>

<div id="seleccionInfo" class="seleccion-activa"></div>

<div class="rutas-panel" id="rutasPanel">
    <h4>Trazado de Rutas (Modo 2D)</h4>
    <div id="unidadesSeleccionadas" class="rutas-lista">
        <div style="color: #888; font-size: 11px; text-align: center;">Selecciona unidades en el mapa 2D</div>
    </div>
    <button id="trazarRutaBtn" class="ruta-boton" onclick="trazarRuta2D()" disabled>Trazar Ruta</button>
    <button id="limpiarRutaBtn" class="ruta-boton limpiar" onclick="limpiarRuta2D()">Limpiar Ruta</button>
</div>

<div class="semaforo">
    <div style="color: white; font-size: 13px; font-weight: bold; margin-bottom: 10px; text-align: center; border-bottom: 1px solid rgba(165,127,44,0.3); padding-bottom: 5px;">
        Semaforo de Avance
    </div>
    <div style="font-size: 11px; color: #A57F2C; margin-bottom: 10px; text-align: center; background: rgba(0,0,0,0.5); padding: 5px; border-radius: 6px;">
        Ano transcurrido: <strong>{pct_dia_esperado:.1f}%</strong>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['verde fuerte']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Verde Fuerte: Avance >= {verde_fuerte_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['verde claro']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Verde Claro: Avance >= {verde_claro_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['amarillo']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Amarillo: Avance >= {amarillo_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['rojo']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Rojo: Avance < {amarillo_umbral:.1f}%</span>
    </div>
</div>

<div class="footer">
    Datos al corte | Avance basado en meta esperada al dia de hoy
</div>

<script>
var unidadesData = {unidades_data_json};
var modo3D = true;
var selectedCluesList = [];
var map2D = null;
var routingControl = null;
var deckgl = null;
var loaderTimeout = null;
var mapLoadingStarted = false;
var currentSelectedClues = null;
var currentSelectedMarker = null;

// Funciones del loader
function showLoader(texto) {{
    var loader = document.getElementById("loaderOverlay");
    var loaderText = document.querySelector(".loader-text");
    if (texto) loaderText.textContent = texto;
    mapLoadingStarted = true;
    void loader.offsetWidth;
    loader.classList.add("active");
    if (loaderTimeout) clearTimeout(loaderTimeout);
    loaderTimeout = setTimeout(function() {{ hideLoader(); }}, 5000);
}}

function hideLoader() {{
    var loader = document.getElementById("loaderOverlay");
    loader.classList.remove("active");
    if (loaderTimeout) {{ clearTimeout(loaderTimeout); loaderTimeout = null; }}
    mapLoadingStarted = false;
}}

// Funciones de resaltado en 3D
function highlightUnit3D(clues) {{
    if (!deckgl) return;
    var selectedUnit = unidadesData.find(u => u.clues === clues);
    if (!selectedUnit) return;
    
    var capaNeon = [
        new deck.ScatterplotLayer({{
            id: "neon-glow",
            data: [selectedUnit],
            getPosition: function(d) {{ return [d.lng, d.lat]; }},
            radiusScale: 1,
            radiusMinPixels: 30,
            radiusMaxPixels: 60,
            getRadius: 400,
            getFillColor: [78, 205, 196, 80],
            stroked: true,
            getLineColor: [78, 205, 196, 200],
            getLineWidth: 3,
            pickable: false
        }}),
        new deck.ScatterplotLayer({{
            id: "neon-core",
            data: [selectedUnit],
            getPosition: function(d) {{ return [d.lng, d.lat]; }},
            radiusScale: 1,
            radiusMinPixels: 15,
            radiusMaxPixels: 30,
            getRadius: 200,
            getFillColor: [78, 205, 196, 180],
            stroked: false,
            pickable: false
        }})
    ];
    
    deckgl.setProps({{
        layers: [
            new deck.ColumnLayer({{
                id: "columnas-avance",
                data: unidadesData,
                diskResolution: 12,
                radius: 400,
                extruded: true,
                pickable: true,
                elevationScale: 0.5,
                getPosition: function(d) {{ return [d.lng, d.lat]; }},
                getElevation: function(d) {{ return d.altura; }},
                getFillColor: function(d) {{
                    if (d.clues === clues) return [255, 215, 0, 255];
                    return [d.r, d.g, d.b, 200];
                }},
                getLineColor: function(d) {{
                    if (d.clues === clues) return [255, 255, 255, 255];
                    return [0, 0, 0, 0];
                }},
                getLineWidth: function(d) {{
                    if (d.clues === clues) return 4;
                    return 0;
                }}
            }}),
            ...capaNeon
        ]
    }});
}}

function resetHighlight3D() {{
    if (!deckgl) return;
    deckgl.setProps({{
        layers: [
            new deck.ColumnLayer({{
                id: "columnas-avance",
                data: unidadesData,
                diskResolution: 12,
                radius: 400,
                extruded: true,
                pickable: true,
                elevationScale: 0.5,
                getPosition: function(d) {{ return [d.lng, d.lat]; }},
                getElevation: function(d) {{ return d.altura; }},
                getFillColor: function(d) {{ return [d.r, d.g, d.b, 200]; }}
            }})
        ]
    }});
}}

function centrarEnUnidad(unidad) {{
    currentSelectedClues = unidad.clues;
    if (modo3D && deckgl) {{
        deckgl.setProps({{ 
            initialViewState: {{ 
                longitude: unidad.lng, latitude: unidad.lat, zoom: 14, pitch: 60, bearing: 0,
                transitionDuration: 1500, transitionInterpolator: new deck.FlyToInterpolator({{ speed: 1.2 }})
            }}
        }});
        setTimeout(function() {{ highlightUnit3D(unidad.clues); }}, 1500);
    }} else if (map2D) {{
        map2D.setView([unidad.lat, unidad.lng], 14);
        setTimeout(function() {{ highlightUnit2D(unidad.clues); }}, 500);
    }}
    buscadorInput.value = unidad.clues;
    resultadosDiv.style.display = "none";
    var seleccionDiv = document.getElementById("seleccionInfo");
    seleccionDiv.innerHTML = "Seleccionado: " + unidad.clues + " - " + unidad.nombre.substring(0, 35);
    seleccionDiv.style.display = "block";
    setTimeout(function() {{ seleccionDiv.style.display = "none"; }}, 3000);
}}

function highlightUnit2D(clues) {{
    if (!map2D) return;
    map2D.eachLayer(function(layer) {{
        if (layer instanceof L.Marker && layer.getPopup()) {{
            var content = layer.getPopup().getContent();
            var match = content.match(/CLUES: (\\S+)/);
            if (match && match[1] === clues) {{
                if (currentSelectedMarker) {{
                    var origIcon = L.divIcon({{
                        className: 'custom-marker',
                        html: '<div style="width: 6px; height: 6px; background-color: ' + currentSelectedMarker.color + '; border-radius: 50%; border: 1px solid #A57F2C;"></div>',
                        iconSize: [6, 6], iconAnchor: [3, 3], popupAnchor: [0, -3]
                    }});
                    currentSelectedMarker.setIcon(origIcon);
                }}
                var unidad = unidadesData.find(u => u.clues === clues);
                var highlightedIcon = L.divIcon({{
                    className: 'custom-marker selected-marker',
                    html: '<div style="width: 16px; height: 16px; background-color: ' + unidad.color_hex + '; border-radius: 50%; border: 3px solid #4ecdc4; box-shadow: 0 0 15px rgba(78,205,196,0.8);"></div>',
                    iconSize: [16, 16], iconAnchor: [8, 8], popupAnchor: [0, -8]
                }});
                layer.setIcon(highlightedIcon);
                currentSelectedMarker = layer;
                currentSelectedMarker.color = unidad.color_hex;
                layer.openPopup();
            }}
        }}
    }});
}}

// Inicializar mapa 2D
function initMap2D() {{
    showLoader("Cargando mapa 2D...");
    setTimeout(function() {{
        try {{
            if (map2D) map2D.remove();
            map2D = L.map('map-2d', {{ attributionControl: false }}).setView([{lat_centro}, {lon_centro}], 6);
            L.tileLayer('https://{{s}}.basemaps.cartocdn.com/dark_all/{{z}}/{{x}}/{{y}}.png', {{
                subdomains: 'abcd', minZoom: 4, maxZoom: 18, attribution: ''
            }}).addTo(map2D);
            
            unidadesData.forEach(function(unidad) {{
                var icon = L.divIcon({{
                    className: 'custom-marker',
                    html: '<div style="width: 6px; height: 6px; background-color: ' + unidad.color_hex + '; border-radius: 50%; border: 1px solid #A57F2C;"></div>',
                    iconSize: [6, 6], iconAnchor: [3, 3], popupAnchor: [0, -3]
                }});
                var marker = L.marker([unidad.lat, unidad.lng], {{ icon: icon }}).addTo(map2D);
                marker.bindPopup(`
                    <div style="font-family: 'Segoe UI', Arial, sans-serif; min-width: 180px;">
                        <strong style="color: #A57F2C;">${{unidad.nombre_de_la_unidad}}</strong><br>
                        <span style="font-size: 11px;">CLUES: ${{unidad.clues}}</span><br>
                        <span style="font-size: 11px;">Avance Real: <span style="color:${{unidad.color_hex}}; font-weight:bold;">${{unidad.avance}}</span></span><br>
                        <span style="font-size: 11px;">Avance Esperado: ${{unidad.pct_dia_fmt}}</span><br>
                        <span style="font-size: 10px; color: #888;">${{unidad.estado_semaforo}}</span>
                    </div>
                `);
                marker.on('click', function(e) {{
                    L.DomEvent.stopPropagation(e);
                    if (selectedCluesList.length < 2 && !selectedCluesList.includes(unidad.clues)) {{
                        selectedCluesList.push(unidad.clues);
                        actualizarPanelSeleccion2D();
                        var newIcon = L.divIcon({{
                            className: 'custom-marker',
                            html: '<div style="width: 10px; height: 10px; background-color: ' + unidad.color_hex + '; border-radius: 50%; border: 2px solid #4ecdc4;"></div>',
                            iconSize: [10, 10], iconAnchor: [5, 5], popupAnchor: [0, -5]
                        }});
                        marker.setIcon(newIcon);
                        setTimeout(() => marker.setIcon(icon), 1500);
                    }} else if (selectedCluesList.length >= 2) {{
                        alert("Solo se pueden seleccionar 2 unidades para la ruta. Use 'Limpiar Ruta' para empezar de nuevo.");
                    }}
                }});
            }});
            if (currentSelectedClues) setTimeout(() => highlightUnit2D(currentSelectedClues), 1000);
            setTimeout(hideLoader, 2000);
        }} catch(e) {{ console.error(e); hideLoader(); }}
    }}, 10);
}}

function actualizarPanelSeleccion2D() {{
    var panel = document.getElementById("unidadesSeleccionadas");
    var btn = document.getElementById("trazarRutaBtn");
    if (selectedCluesList.length === 0) {{
        panel.innerHTML = '<div style="color:#888;text-align:center;">Selecciona unidades en el mapa 2D</div>';
        btn.disabled = true;
    }} else if (selectedCluesList.length === 1) {{
        var u = unidadesData.find(d => d.clues === selectedCluesList[0]);
        panel.innerHTML = '<div class="ruta-item"><div class="ruta-origen-destino">Origen: ' + u.clues + '</div><div>' + u.nombre.substring(0,40) + '</div></div><div style="color:#A57F2C;text-align:center;">Selecciona destino</div>';
        btn.disabled = true;
    }} else {{
        var o = unidadesData.find(d => d.clues === selectedCluesList[0]);
        var d = unidadesData.find(d => d.clues === selectedCluesList[1]);
        panel.innerHTML = '<div class="ruta-item"><div class="ruta-origen-destino">Origen: ' + o.clues + '</div><div>' + o.nombre.substring(0,35) + '</div></div><div class="ruta-item"><div class="ruta-origen-destino">Destino: ' + d.clues + '</div><div>' + d.nombre.substring(0,35) + '</div></div>';
        btn.disabled = false;
    }}
}}

function trazarRuta2D() {{
    if (selectedCluesList.length !== 2) return;
    var o = unidadesData.find(d => d.clues === selectedCluesList[0]);
    var d = unidadesData.find(d => d.clues === selectedCluesList[1]);
    if (!o || !d) return;
    if (routingControl) map2D.removeControl(routingControl);
    routingControl = L.Routing.control({{
        waypoints: [L.latLng(o.lat, o.lng), L.latLng(d.lat, d.lng)],
        routeWhileDragging: false, showAlternatives: false,
        lineOptions: {{ styles: [{{ color: '#4ecdc4', weight: 5, opacity: 0.8 }}] }},
        createMarker: () => null,
        router: L.Routing.osrmv1({{ serviceUrl: 'https://router.project-osrm.org/route/v1' }}),
        show: false
    }}).addTo(map2D);
    routingControl.on('routesfound', function(e) {{
        var route = e.routes[0];
        var dist = (route.summary.totalDistance / 1000).toFixed(1);
        var dur = Math.round(route.summary.totalTime / 60);
        L.popup().setLatLng([(o.lat+d.lat)/2, (o.lng+d.lng)/2]).setContent('<div class="ruta-popup"><div class="distancia">' + dist + ' km</div><div class="tiempo">' + dur + ' minutos</div></div>').openOn(map2D);
        map2D.fitBounds(L.latLngBounds([o.lat, o.lng], [d.lat, d.lng]), {{ padding: [50,50] }});
    }});
}}

function limpiarRuta2D() {{
    selectedCluesList = [];
    if (routingControl) map2D.removeControl(routingControl);
    routingControl = null;
    actualizarPanelSeleccion2D();
    if (map2D) map2D.closePopup();
}}

function toggleModo() {{
    var map3d = document.getElementById("map-3d");
    var map2dDiv = document.getElementById("map-2d");
    var rutasPanel = document.getElementById("rutasPanel");
    var btn = document.querySelector(".btn-toggle");
    modo3D = !modo3D;
    if (modo3D) {{
        showLoader("Cargando mapa 3D...");
        map3d.style.display = "block";
        map2dDiv.style.display = "none";
        rutasPanel.style.display = "none";
        btn.textContent = "Cambiar a Modo 2D (Rutas)";
        selectedCluesList = [];
        if (routingControl) map2D.removeControl(routingControl);
        setTimeout(() => {{
            if (currentSelectedClues) highlightUnit3D(currentSelectedClues);
            setTimeout(hideLoader, 800);
        }}, 100);
    }} else {{
        showLoader("Cargando mapa 2D...");
        map3d.style.display = "none";
        map2dDiv.style.display = "block";
        rutasPanel.style.display = "block";
        btn.textContent = "Cambiar a Modo 3D";
        resetHighlight3D();
        if (map2D) {{
            map2D.invalidateSize();
            if (currentSelectedClues) setTimeout(() => highlightUnit2D(currentSelectedClues), 500);
            setTimeout(hideLoader, 1500);
        }} else {{
            initMap2D();
        }}
    }}
}}

function initMap3D() {{
    deckgl = new deck.DeckGL({{
        container: "map-3d",
        mapStyle: "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
        initialViewState: {{ longitude: {lon_centro}, latitude: {lat_centro}, zoom: 5.5, pitch: 55, bearing: 0 }},
        controller: true,
        getTooltip: function(info) {{
            if (!info.object) return null;
            var d = info.object;
            return {{
                html: '<div style="font-family: Segoe UI, Arial, sans-serif; min-width:260px; padding: 10px;">' +
                    '<div style="font-weight:bold;font-size:13px;margin-bottom:6px;border-bottom: 1px solid rgba(165,127,44,0.3);padding-bottom: 4px;color:#A57F2C;">' + d.nombre_de_la_unidad + '</div>' +
                    '<div style="display: grid; grid-template-columns: 85px 1fr; gap: 4px; font-size: 11px;">' +
                    '<span style="color:#aaa;">CLUES:</span><span style="font-weight:500;">' + d.clues + '</span>' +
                    '<span style="color:#aaa;">Entidad:</span><span>' + d.entidad + '</span>' +
                    '<span style="color:#aaa;">Nivel:</span><span>' + d.nivel + '</span>' +
                    '<span style="color:#aaa;">Indicador:</span><span>' + d.tipo + '</span>' +
                    '<span style="color:#aaa;">Realizado:</span><span style="font-weight:500;">' + d.realizado + '</span>' +
                    '<span style="color:#aaa;">Meta Anual:</span><span>' + d.meta_anual + '</span>' +
                    '<span style="color:#aaa;">Meta Esperada:</span><span>' + d.meta_esperada + '</span>' +
                    '<span style="color:#aaa;">Avance Real:</span><span style="color:' + d.color_hex + '; font-weight:bold;">' + d.avance + '</span>' +
                    '<span style="color:#aaa;">Avance Esperado:</span><span>' + d.pct_dia_fmt + '</span>' +
                    '</div>' +
                    '<div style="margin-top: 6px; padding-top: 4px; border-top: 1px solid rgba(165,127,44,0.2); font-size: 10px; color: #A57F2C;">' + d.estado_semaforo + '</div></div>'
            }};
        }},
        layers: [
            new deck.ColumnLayer({{
                id: "columnas-avance",
                data: unidadesData,
                diskResolution: 12,
                radius: 400,
                extruded: true,
                pickable: true,
                elevationScale: 0.5,
                getPosition: d => [d.lng, d.lat],
                getElevation: d => d.altura,
                getFillColor: d => [d.r, d.g, d.b, 200]
            }})
        ]
    }});
    deckgl.canvas.addEventListener('click', function(event) {{
        var pick = deckgl.pickObject({{x: event.clientX, y: event.clientY}});
        if (pick && pick.object) {{
            if (currentSelectedClues === pick.object.clues) {{
                currentSelectedClues = null;
                resetHighlight3D();
            }} else {{
                currentSelectedClues = pick.object.clues;
                highlightUnit3D(currentSelectedClues);
            }}
        }}
    }});
}}

// Buscador
var buscadorInput = document.getElementById("buscadorClues");
var resultadosDiv = document.getElementById("resultadosBusqueda");

function buscarUnidades(texto) {{
    if (texto.length < 2) {{ resultadosDiv.style.display = "none"; return []; }}
    texto = texto.toLowerCase();
    return unidadesData.filter(u => u.clues.toLowerCase().includes(texto) || u.nombre.toLowerCase().includes(texto)).slice(0,10);
}}

function mostrarResultados(resultados) {{
    resultadosDiv.innerHTML = "";
    if (resultados.length === 0) {{ resultadosDiv.style.display = "none"; return; }}
    resultados.forEach(u => {{
        var item = document.createElement("div");
        item.className = "resultado-item";
        item.innerHTML = '<div style="font-weight:bold;color:#A57F2C;font-size:13px;">' + u.clues + '</div><div style="font-size:11px;color:white;">' + u.nombre.substring(0,50) + '</div><div style="font-size:10px;color:#A57F2C;">Avance: ' + u.avance + '</div>';
        item.onclick = () => centrarEnUnidad(u);
        resultadosDiv.appendChild(item);
    }});
    resultadosDiv.style.display = "block";
}}

buscadorInput.addEventListener("input", e => mostrarResultados(buscarUnidades(e.target.value)));
document.addEventListener("click", e => {{
    if (!buscadorInput.contains(e.target) && !resultadosDiv.contains(e.target)) resultadosDiv.style.display = "none";
}});

initMap3D();
</script>
</body>
</html>
"""

# ============================================================
# GUARDAR Y ABRIR
# ============================================================

archivo_salida = Path("index.html")

with open(archivo_salida, "w", encoding="utf-8") as f:
    f.write(html_completo)

webbrowser.open(archivo_salida.resolve().as_uri())

True

In [267]:
# ============================================================
# HTML COMPLETO CON 2D Y 3D - CON CLIC EN MAPA 2D PARA DISTANCIAS
# ============================================================

html_completo = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<title>Hospital Dashboard - Modo 3D y 2D para Rutas</title>

<script src="https://unpkg.com/deck.gl@latest/dist.min.js"></script>
<script src="https://unpkg.com/maplibre-gl@latest/dist/maplibre-gl.js"></script>
<link href="https://unpkg.com/maplibre-gl@latest/dist/maplibre-gl.css" rel="stylesheet"/>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://unpkg.com/leaflet-routing-machine@latest/dist/leaflet-routing-machine.js"></script>
<link rel="stylesheet" href="https://unpkg.com/leaflet-routing-machine@latest/dist/leaflet-routing-machine.css" />

<style>
html, body {{
    margin: 0;
    width: 100%;
    height: 100%;
    font-family: 'Segoe UI', Arial, sans-serif;
    background: #0a0a0a;
}}

#map-3d {{
    width: 100%;
    height: 100%;
    background: #0d0d0d;
    display: block;
}}

#map-2d {{
    width: 100%;
    height: 100%;
    background: #0d0d0d;
    display: none;
}}

.leaflet-control-attribution {{
    display: none !important;
}}

/* LOADER */
.loader-overlay {{
    position: fixed;
    top: 0;
    left: 0;
    width: 100%;
    height: 100%;
    background: rgba(0,0,0,0.9);
    backdrop-filter: blur(8px);
    z-index: 20000;
    display: flex;
    align-items: center;
    justify-content: center;
    flex-direction: column;
    opacity: 0;
    visibility: hidden;
    transition: opacity 0.2s ease, visibility 0s linear 0.3s;
    pointer-events: none;
}}

.loader-overlay.active {{
    opacity: 1;
    visibility: visible;
    transition: opacity 0.2s ease, visibility 0s linear 0s;
    pointer-events: all;
}}

.loader-text {{
    color: #A57F2C;
    margin-top: 30px;
    font-size: 14px;
    font-weight: bold;
    letter-spacing: 2px;
    text-transform: uppercase;
    background: rgba(0,0,0,0.6);
    padding: 8px 20px;
    border-radius: 30px;
    border: 1px solid rgba(165,127,44,0.3);
}}

.loader-svg {{
    width: 450px;
    height: 300px;
    animation: spin 5s linear infinite;
    filter: drop-shadow(0 0 15px rgba(165,127,44,0.6));
}}

@keyframes spin {{
    0% {{ transform: rotate(0deg); }}
    100% {{ transform: rotate(360deg); }}
}}

.header {{
    position: fixed;
    top: 0;
    left: 0;
    right: 0;
    z-index: 10000;
    background: #1E5B4F;
    color: white;
    padding: 12px 25px;
    font-size: 18px;
    font-weight: bold;
    text-align: center;
    box-shadow: 0 2px 10px rgba(0,0,0,0.5);
    pointer-events: none;
    border-bottom: 1px solid rgba(165,127,44,0.3);
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 15px;
}}

.header img {{
    height: 40px;
    width: auto;
    filter: brightness(0) invert(1);
}}

.btn-toggle {{
    position: fixed;
    top: 80px;
    right: 20px;
    z-index: 10001;
    background: linear-gradient(135deg, #A57F2C, #1E5B4F);
    color: white;
    border: none;
    padding: 12px 24px;
    border-radius: 30px;
    font-size: 14px;
    font-weight: bold;
    cursor: pointer;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-shadow: 0 4px 15px rgba(0,0,0,0.3);
    transition: all 0.3s;
    pointer-events: auto;
}}

.btn-toggle:hover {{
    transform: translateY(-2px);
    box-shadow: 0 6px 20px rgba(165,127,44,0.4);
}}

.kpi-container {{
    position: fixed;
    top: 70px;
    left: 0;
    right: 0;
    z-index: 10000;
    display: grid;
    grid-template-columns: repeat(5, 1fr);
    gap: 12px;
    padding: 12px 20px;
    pointer-events: none;
}}

.kpi-card {{
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    padding: 10px 15px;
    border-radius: 10px;
    border: 1px solid rgba(165,127,44,0.3);
    pointer-events: auto;
    box-shadow: 0 2px 5px rgba(0,0,0,0.3);
}}

.kpi-label {{
    font-size: 10px;
    color: #A57F2C;
    text-transform: uppercase;
    letter-spacing: 1px;
}}

.kpi-value {{
    font-size: 24px;
    font-weight: bold;
    color: white;
}}

.kpi-sub {{
    font-size: 9px;
    color: #888;
}}

.dashboard-btn {{
    position: fixed;
    top: 380px;
    left: 20px;
    z-index: 10001;
}}

.dashboard-btn button {{
    background: linear-gradient(135deg, #A57F2C, #1E5B4F);
    color: white;
    border: none;
    padding: 8px 18px;
    border-radius: 25px;
    font-size: 12px;
    font-weight: bold;
    cursor: pointer;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-shadow: 0 2px 5px rgba(0,0,0,0.3);
    transition: all 0.3s;
    pointer-events: auto;
}}

.dashboard-btn button:hover {{
    transform: translateY(-1px);
    box-shadow: 0 4px 10px rgba(165,127,44,0.3);
}}

.buscador {{
    position: fixed;
    top: 200px;
    left: 20px;
    z-index: 10001;
    width: 320px;
}}

.buscador-box {{
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    border-radius: 12px;
    padding: 12px;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
}}

.buscador-label {{
    color: #A57F2C;
    font-size: 10px;
    text-transform: uppercase;
    margin-bottom: 8px;
    letter-spacing: 1px;
}}

.buscador-input {{
    width: 100%;
    padding: 10px 15px;
    background: rgba(20,20,20,0.9);
    border: 1px solid rgba(165,127,44,0.3);
    border-radius: 8px;
    color: white;
    font-size: 13px;
    outline: none;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-sizing: border-box;
}}

.buscador-input:focus {{
    border-color: #A57F2C;
    box-shadow: 0 0 5px rgba(165,127,44,0.5);
}}

.resultados {{
    background: rgba(10,10,10,0.95);
    border-radius: 12px;
    margin-top: 8px;
    max-height: 350px;
    overflow-y: auto;
    display: none;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
}}

.resultado-item {{
    padding: 12px 15px;
    cursor: pointer;
    border-bottom: 1px solid rgba(165,127,44,0.1);
    font-family: 'Segoe UI', Arial, sans-serif;
    transition: all 0.2s;
}}

.resultado-item:hover {{
    background: rgba(165,127,44,0.15);
    border-left: 3px solid #A57F2C;
}}

.rutas-panel {{
    position: fixed;
    bottom: 20px;
    left: 20px;
    z-index: 10001;
    background: rgba(10,10,10,0.95);
    backdrop-filter: blur(10px);
    border-radius: 12px;
    min-width: 280px;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    pointer-events: auto;
    padding: 12px;
    display: none;
}}

.rutas-panel h4 {{
    margin: 0 0 10px 0;
    color: #A57F2C;
    font-size: 13px;
    text-align: center;
    border-bottom: 1px solid rgba(165,127,44,0.3);
    padding-bottom: 6px;
}}

.rutas-lista {{
    margin-bottom: 10px;
}}

.ruta-item {{
    background: rgba(30,30,30,0.8);
    border-radius: 8px;
    padding: 8px;
    margin-bottom: 8px;
    font-size: 11px;
    cursor: pointer;
    transition: all 0.2s;
}}

.ruta-item:hover {{
    background: rgba(165,127,44,0.2);
    border-left: 3px solid #4ecdc4;
}}

.ruta-origen-destino {{
    font-weight: bold;
    color: #4ecdc4;
    margin-bottom: 4px;
}}

.ruta-info {{
    color: #aaa;
    font-size: 10px;
    display: flex;
    justify-content: space-between;
}}

.ruta-boton {{
    width: 100%;
    background: linear-gradient(135deg, #1E5B4F, #A57F2C);
    color: white;
    border: none;
    padding: 8px;
    border-radius: 8px;
    font-size: 11px;
    font-weight: bold;
    cursor: pointer;
    margin-top: 5px;
    transition: all 0.2s;
}}

.ruta-boton:hover {{
    transform: translateY(-1px);
    box-shadow: 0 2px 8px rgba(165,127,44,0.4);
}}

.ruta-boton.limpiar {{
    background: rgba(100,100,100,0.6);
    margin-top: 5px;
}}

.ruta-boton.limpiar:hover {{
    background: rgba(150,150,150,0.8);
}}

.seleccion-activa {{
    position: fixed;
    top: 420px;
    left: 20px;
    z-index: 10001;
    background: rgba(165,127,44,0.2);
    backdrop-filter: blur(10px);
    padding: 5px 12px;
    border-radius: 20px;
    border: 1px solid #A57F2C;
    color: #A57F2C;
    font-size: 11px;
    font-weight: bold;
    pointer-events: none;
    display: none;
}}

.semaforo {{
    position: fixed;
    bottom: 20px;
    right: 20px;
    z-index: 10001;
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    padding: 12px 18px;
    border-radius: 12px;
    min-width: 240px;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    pointer-events: auto;
}}

.footer {{
    position: fixed;
    bottom: 20px;
    left: 320px;
    z-index: 10001;
    background: rgba(10,10,10,0.8);
    backdrop-filter: blur(10px);
    padding: 8px 12px;
    border-radius: 8px;
    font-size: 10px;
    color: #888;
    border: 1px solid rgba(165,127,44,0.2);
    pointer-events: auto;
}}

.deck-tooltip {{
    background: rgba(0,0,0,0.85) !important;
    backdrop-filter: blur(8px) !important;
    border: 1px solid rgba(165,127,44,0.4) !important;
    border-radius: 8px !important;
    box-shadow: 0 4px 15px rgba(0,0,0,0.5) !important;
    padding: 0 !important;
}}

.leaflet-popup-content-wrapper {{
    background: rgba(10,10,10,0.95);
    backdrop-filter: blur(10px);
    border: 1px solid rgba(165,127,44,0.3);
    border-radius: 12px;
    color: white;
    min-width: 200px;
}}

.leaflet-popup-tip {{
    background: rgba(10,10,10,0.95);
    border: 1px solid rgba(165,127,44,0.3);
}}

.leaflet-popup-content {{
    margin: 8px 12px;
    line-height: 1.4;
}}

.ruta-popup {{
    font-family: 'Segoe UI', Arial, sans-serif;
    text-align: center;
}}

.ruta-popup .distancia {{
    font-size: 16px;
    font-weight: bold;
    color: #4ecdc4;
    margin-bottom: 5px;
}}

.ruta-popup .tiempo {{
    font-size: 13px;
    color: #A57F2C;
}}

.custom-marker {{
    background: transparent;
    border: none;
}}

.custom-marker div {{
    width: 6px;
    height: 6px;
    border-radius: 50%;
    border: 1px solid rgba(0,0,0,0.3);
    box-shadow: 0 0 2px rgba(0,0,0,0.5);
    transition: all 0.2s ease;
}}

.custom-marker div:hover {{
    transform: scale(1.5);
    box-shadow: 0 0 4px rgba(165,127,44,0.8);
}}

.selected-marker div {{
    width: 16px !important;
    height: 16px !important;
    border: 3px solid #4ecdc4 !important;
    box-shadow: 0 0 15px rgba(78, 205, 196, 0.8) !important;
    animation: pulse 1.5s ease-in-out infinite !important;
}}

@keyframes pulse {{
    0% {{ transform: scale(1); opacity: 1; }}
    50% {{ transform: scale(1.3); opacity: 0.7; }}
    100% {{ transform: scale(1); opacity: 1; }}
}}

.leaflet-routing-container {{
    display: none !important;
}}

.distancia-popup .leaflet-popup-content-wrapper {{
    max-width: 350px;
    min-width: 280px;
}}

.distancia-popup .leaflet-popup-content {{
    margin: 8px 12px;
}}

.distancia-popup .leaflet-popup-tip {{
    background: rgba(10,10,10,0.95);
}}

.distancia-popup div::-webkit-scrollbar {{
    width: 6px;
}}

.distancia-popup div::-webkit-scrollbar-track {{
    background: rgba(30,30,30,0.5);
    border-radius: 3px;
}}

.distancia-popup div::-webkit-scrollbar-thumb {{
    background: #A57F2C;
    border-radius: 3px;
}}

.distancia-popup div::-webkit-scrollbar-thumb:hover {{
    background: #4ecdc4;
}}
</style>
</head>
<body>

<div id="map-3d"></div>
<div id="map-2d"></div>

<div id="loaderOverlay" class="loader-overlay">
    <img class="loader-svg" src="{svg_data_uri}" alt="Cargando...">
    <div class="loader-text">Cargando mapa...</div>
</div>

<button class="btn-toggle" onclick="toggleModo()">Cambiar a Modo 2D (Rutas)</button>

<div class="header">
    <img src="https://imssbienestar.gob.mx/assets/img/imb_b.svg" 
         alt="IMSS Bienestar" 
         onerror="this.style.display='none'">
    Hospital Dashboard - Productividad por Unidad Medica
</div>

<div class="kpi-container">
    <div class="kpi-card">
        <div class="kpi-label">Unidades Activas</div>
        <div class="kpi-value">{total_unidades_activas:,}</div>
        <div class="kpi-sub">Cirugia: {unidades_cirugias_activas} | Consulta: {unidades_consultas_activas}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Avance General</div>
        <div class="kpi-value">{avg_avance_general:.1f}%</div>
        <div class="kpi-sub">Cirugia: {avg_avance_cirugias:.1f}% | Consulta: {avg_avance_consultas:.1f}%</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Volumen Realizado - Cirugia</div>
        <div class="kpi-value" style="color: #ffffff;">{volumen_cirugias:,.0f}</div>
        <div class="kpi-sub">Total de cirugias realizadas</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Volumen Realizado - Consulta</div>
        <div class="kpi-value" style="color: #4ecdc4;">{volumen_consultas:,.0f}</div>
        <div class="kpi-sub" style="font-size: 8px;">General: {consulta_general:,.0f} | Especialidad: {consulta_especialidad:,.0f}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Cobertura</div>
        <div class="kpi-value">{entidades_unicas}</div>
        <div class="kpi-sub">Entidades Federativas</div>
    </div>
</div>

<div class="dashboard-btn">
    <button onclick="window.open('https://argontc.shinyapps.io/pptx/', '_blank')">
        Ver Dashboard de Reportes
    </button>
</div>

<div class="buscador">
    <div class="buscador-box">
        <div class="buscador-label">Buscador de Unidades</div>
        <input type="text" id="buscadorClues" class="buscador-input" placeholder="Buscar por CLUES o nombre...">
    </div>
    <div id="resultadosBusqueda" class="resultados"></div>
</div>

<div id="seleccionInfo" class="seleccion-activa"></div>

<div class="rutas-panel" id="rutasPanel">
    <h4>Trazado de Rutas (Modo 2D)</h4>
    <div id="unidadesSeleccionadas" class="rutas-lista">
        <div style="color: #888; font-size: 11px; text-align: center;">Selecciona unidades en el mapa 2D</div>
    </div>
    <button id="trazarRutaBtn" class="ruta-boton" onclick="trazarRuta2D()" disabled>Trazar Ruta</button>
    <button id="limpiarRutaBtn" class="ruta-boton limpiar" onclick="limpiarRuta2D()">Limpiar Ruta</button>
</div>

<div class="semaforo">
    <div style="color: white; font-size: 13px; font-weight: bold; margin-bottom: 10px; text-align: center; border-bottom: 1px solid rgba(165,127,44,0.3); padding-bottom: 5px;">
        Semaforo de Avance
    </div>
    <div style="font-size: 11px; color: #A57F2C; margin-bottom: 10px; text-align: center; background: rgba(0,0,0,0.5); padding: 5px; border-radius: 6px;">
        Ano transcurrido: <strong>{pct_dia_esperado:.1f}%</strong>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['verde fuerte']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Verde Fuerte: Avance >= {verde_fuerte_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['verde claro']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Verde Claro: Avance >= {verde_claro_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['amarillo']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Amarillo: Avance >= {amarillo_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['rojo']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Rojo: Avance < {amarillo_umbral:.1f}%</span>
    </div>
</div>

<div class="footer">
    Datos al corte | Avance basado en meta esperada al dia de hoy
</div>

<script>
var unidadesData = {unidades_data_json};
var modo3D = true;
var selectedCluesList = [];
var map2D = null;
var routingControl = null;
var deckgl = null;
var loaderTimeout = null;
var mapLoadingStarted = false;
var currentSelectedClues = null;
var currentSelectedMarker = null;

function showLoader(texto) {{
    var loader = document.getElementById("loaderOverlay");
    var loaderText = document.querySelector(".loader-text");
    if (texto) loaderText.textContent = texto;
    mapLoadingStarted = true;
    void loader.offsetWidth;
    loader.classList.add("active");
    if (loaderTimeout) clearTimeout(loaderTimeout);
    loaderTimeout = setTimeout(function() {{ hideLoader(); }}, 5000);
}}

function hideLoader() {{
    var loader = document.getElementById("loaderOverlay");
    loader.classList.remove("active");
    if (loaderTimeout) {{ clearTimeout(loaderTimeout); loaderTimeout = null; }}
    mapLoadingStarted = false;
}}

function highlightUnit3D(clues) {{
    if (!deckgl) return;
    var selectedUnit = unidadesData.find(u => u.clues === clues);
    if (!selectedUnit) return;
    
    var capaNeon = [
        new deck.ScatterplotLayer({{
            id: "neon-glow",
            data: [selectedUnit],
            getPosition: function(d) {{ return [d.lng, d.lat]; }},
            radiusScale: 1,
            radiusMinPixels: 30,
            radiusMaxPixels: 60,
            getRadius: 400,
            getFillColor: [78, 205, 196, 80],
            stroked: true,
            getLineColor: [78, 205, 196, 200],
            getLineWidth: 3,
            pickable: false
        }}),
        new deck.ScatterplotLayer({{
            id: "neon-core",
            data: [selectedUnit],
            getPosition: function(d) {{ return [d.lng, d.lat]; }},
            radiusScale: 1,
            radiusMinPixels: 15,
            radiusMaxPixels: 30,
            getRadius: 200,
            getFillColor: [78, 205, 196, 180],
            stroked: false,
            pickable: false
        }})
    ];
    
    deckgl.setProps({{
        layers: [
            new deck.ColumnLayer({{
                id: "columnas-avance",
                data: unidadesData,
                diskResolution: 12,
                radius: 400,
                extruded: true,
                pickable: true,
                elevationScale: 0.5,
                getPosition: function(d) {{ return [d.lng, d.lat]; }},
                getElevation: function(d) {{ return d.altura; }},
                getFillColor: function(d) {{
                    if (d.clues === clues) return [255, 215, 0, 255];
                    return [d.r, d.g, d.b, 200];
                }},
                getLineColor: function(d) {{
                    if (d.clues === clues) return [255, 255, 255, 255];
                    return [0, 0, 0, 0];
                }},
                getLineWidth: function(d) {{
                    if (d.clues === clues) return 4;
                    return 0;
                }}
            }}),
            ...capaNeon
        ]
    }});
}}

function resetHighlight3D() {{
    if (!deckgl) return;
    deckgl.setProps({{
        layers: [
            new deck.ColumnLayer({{
                id: "columnas-avance",
                data: unidadesData,
                diskResolution: 12,
                radius: 400,
                extruded: true,
                pickable: true,
                elevationScale: 0.5,
                getPosition: function(d) {{ return [d.lng, d.lat]; }},
                getElevation: function(d) {{ return d.altura; }},
                getFillColor: function(d) {{ return [d.r, d.g, d.b, 200]; }}
            }})
        ]
    }});
}}

function centrarEnUnidad(unidad) {{
    currentSelectedClues = unidad.clues;
    if (modo3D && deckgl) {{
        deckgl.setProps({{ 
            initialViewState: {{ 
                longitude: unidad.lng, latitude: unidad.lat, zoom: 14, pitch: 60, bearing: 0,
                transitionDuration: 1500, transitionInterpolator: new deck.FlyToInterpolator({{ speed: 1.2 }})
            }}
        }});
        setTimeout(function() {{ highlightUnit3D(unidad.clues); }}, 1500);
    }} else if (map2D) {{
        map2D.setView([unidad.lat, unidad.lng], 14);
        setTimeout(function() {{ highlightUnit2D(unidad.clues); }}, 500);
    }}
    buscadorInput.value = unidad.clues;
    resultadosDiv.style.display = "none";
    var seleccionDiv = document.getElementById("seleccionInfo");
    seleccionDiv.innerHTML = "Seleccionado: " + unidad.clues + " - " + unidad.nombre.substring(0, 35);
    seleccionDiv.style.display = "block";
    setTimeout(function() {{ seleccionDiv.style.display = "none"; }}, 3000);
}}

function highlightUnit2D(clues) {{
    if (!map2D) return;
    map2D.eachLayer(function(layer) {{
        if (layer instanceof L.Marker && layer.getPopup()) {{
            var content = layer.getPopup().getContent();
            var match = content.match(/CLUES: (\\S+)/);
            if (match && match[1] === clues) {{
                if (currentSelectedMarker) {{
                    var origIcon = L.divIcon({{
                        className: 'custom-marker',
                        html: '<div style="width: 6px; height: 6px; background-color: ' + currentSelectedMarker.color + '; border-radius: 50%; border: 1px solid #A57F2C;"></div>',
                        iconSize: [6, 6], iconAnchor: [3, 3], popupAnchor: [0, -3]
                    }});
                    currentSelectedMarker.setIcon(origIcon);
                }}
                var unidad = unidadesData.find(u => u.clues === clues);
                var highlightedIcon = L.divIcon({{
                    className: 'custom-marker selected-marker',
                    html: '<div style="width: 16px; height: 16px; background-color: ' + unidad.color_hex + '; border-radius: 50%; border: 3px solid #4ecdc4; box-shadow: 0 0 15px rgba(78,205,196,0.8);"></div>',
                    iconSize: [16, 16], iconAnchor: [8, 8], popupAnchor: [0, -8]
                }});
                layer.setIcon(highlightedIcon);
                currentSelectedMarker = layer;
                currentSelectedMarker.color = unidad.color_hex;
                layer.openPopup();
            }}
        }}
    }});
}}

function calcularDistanciaAPuntosRelevantes(lat, lng) {{
    var distancias = [];
    
    for (var i = 0; i < unidadesData.length; i++) {{
        var unidad = unidadesData[i];
        var R = 6371;
        var dLat = (unidad.lat - lat) * Math.PI / 180;
        var dLon = (unidad.lng - lng) * Math.PI / 180;
        var a = Math.sin(dLat/2) * Math.sin(dLat/2) +
                Math.cos(lat * Math.PI / 180) * Math.cos(unidad.lat * Math.PI / 180) *
                Math.sin(dLon/2) * Math.sin(dLon/2);
        var c = 2 * Math.atan2(Math.sqrt(a), Math.sqrt(1-a));
        var distancia = R * c;
        
        distancias.push({{
            unidad: unidad,
            distancia_km: distancia,
            distancia_metros: distancia * 1000
        }});
    }}
    
    distancias.sort(function(a, b) {{
        return a.distancia_km - b.distancia_km;
    }});
    
    return distancias.slice(0, 5);
}}

function mostrarPopupDistancias(lat, lng) {{
    var distancias = calcularDistanciaAPuntosRelevantes(lat, lng);
    
    var popupContent = '<div style="font-family: Segoe UI, Arial, sans-serif; min-width: 280px; max-width: 320px;">' +
        '<div style="background: linear-gradient(135deg, #A57F2C, #1E5B4F); padding: 8px; border-radius: 8px; margin: -8px -12px 8px -12px; text-align: center; color: white;">' +
        '<strong>Distancias desde este punto</strong></div>' +
        '<div style="margin-bottom: 10px; padding: 6px; background: rgba(165,127,44,0.1); border-radius: 6px; text-align: center;">' +
        '<span style="color: #A57F2C;">Coordenadas:</span><br>' +
        '<strong>' + lat.toFixed(6) + '°, ' + lng.toFixed(6) + '°</strong></div>' +
        '<div style="max-height: 250px; overflow-y: auto;">';
    
    for (var i = 0; i < distancias.length; i++) {{
        var item = distancias[i];
        var unidad = item.unidad;
        var distancia_km = item.distancia_km;
        var distancia_metros = item.distancia_metros;
        
        var distancia_texto;
        if (distancia_km < 1) {{
            distancia_texto = Math.round(distancia_metros) + ' metros';
        }} else {{
            distancia_texto = distancia_km.toFixed(2) + ' km';
        }}
        
        var unidadDataStr = JSON.stringify(unidad).replace(/"/g, '&quot;');
        
        popupContent += '<div style="margin-bottom: 10px; padding: 8px; background: rgba(30,30,30,0.8); border-radius: 6px; border-left: 3px solid ' + unidad.color_hex + '; cursor: pointer;" onclick="centrarEnUnidad(' + unidadDataStr + ')">' +
            '<div style="display: flex; justify-content: space-between; align-items: center;">' +
            '<div><div style="font-weight: bold; color: #A57F2C; font-size: 12px;">' + unidad.clues + '</div>' +
            '<div style="font-size: 10px; color: #ccc; margin-top: 2px;">' + unidad.nombre.substring(0, 35) + '...</div></div>' +
            '<div style="text-align: right;"><div style="font-size: 14px; font-weight: bold; color: #4ecdc4;">' + distancia_texto + '</div>' +
            '<div style="font-size: 9px; color: #888;">Avance: ' + unidad.avance + '</div></div></div></div>';
    }}
    
    popupContent += '</div><div style="margin-top: 8px; padding-top: 6px; border-top: 1px solid rgba(165,127,44,0.3); font-size: 10px; text-align: center; color: #888;">' +
        'Click en cualquier unidad para centrar el mapa</div></div>';
    
    L.popup({{
        maxWidth: 350,
        minWidth: 280,
        className: 'distancia-popup'
    }})
    .setLatLng([lat, lng])
    .setContent(popupContent)
    .openOn(map2D);
}}

function initMap2D() {{
    showLoader("Cargando mapa 2D...");
    setTimeout(function() {{
        try {{
            if (map2D) map2D.remove();
            map2D = L.map('map-2d', {{ attributionControl: false }}).setView([{lat_centro}, {lon_centro}], 6);
            L.tileLayer('https://{{s}}.basemaps.cartocdn.com/dark_all/{{z}}/{{x}}/{{y}}.png', {{
                subdomains: 'abcd', minZoom: 4, maxZoom: 18, attribution: ''
            }}).addTo(map2D);
            
            map2D.on('click', function(e) {{
                var clickedOnMarker = false;
                map2D.eachLayer(function(layer) {{
                    if (layer instanceof L.Marker) {{
                        var markerLatLng = layer.getLatLng();
                        var distance = Math.sqrt(Math.pow(e.latlng.lat - markerLatLng.lat, 2) + Math.pow(e.latlng.lng - markerLatLng.lng, 2));
                        if (distance < 0.0001) {{
                            clickedOnMarker = true;
                        }}
                    }}
                }});
                
                if (!clickedOnMarker) {{
                    mostrarPopupDistancias(e.latlng.lat, e.latlng.lng);
                }}
            }});
            
            for (var i = 0; i < unidadesData.length; i++) {{
                var unidad = unidadesData[i];
                var icon = L.divIcon({{
                    className: 'custom-marker',
                    html: '<div style="width: 6px; height: 6px; background-color: ' + unidad.color_hex + '; border-radius: 50%; border: 1px solid #A57F2C;"></div>',
                    iconSize: [6, 6], iconAnchor: [3, 3], popupAnchor: [0, -3]
                }});
                var marker = L.marker([unidad.lat, unidad.lng], {{ icon: icon }}).addTo(map2D);
                marker.bindPopup('<div style="font-family: Segoe UI, Arial, sans-serif; min-width: 180px;">' +
                    '<strong style="color: #A57F2C;">' + unidad.nombre_de_la_unidad + '</strong><br>' +
                    '<span style="font-size: 11px;">CLUES: ' + unidad.clues + '</span><br>' +
                    '<span style="font-size: 11px;">Avance Real: <span style="color:' + unidad.color_hex + '; font-weight:bold;">' + unidad.avance + '</span></span><br>' +
                    '<span style="font-size: 11px;">Avance Esperado: ' + unidad.pct_dia_fmt + '</span><br>' +
                    '<span style="font-size: 10px; color: #888;">' + unidad.estado_semaforo + '</span></div>');
                marker.on('click', (function(unidad, icon) {{
                    return function(e) {{
                        L.DomEvent.stopPropagation(e);
                        if (selectedCluesList.length < 2 && selectedCluesList.indexOf(unidad.clues) === -1) {{
                            selectedCluesList.push(unidad.clues);
                            actualizarPanelSeleccion2D();
                            var newIcon = L.divIcon({{
                                className: 'custom-marker',
                                html: '<div style="width: 10px; height: 10px; background-color: ' + unidad.color_hex + '; border-radius: 50%; border: 2px solid #4ecdc4;"></div>',
                                iconSize: [10, 10], iconAnchor: [5, 5], popupAnchor: [0, -5]
                            }});
                            marker.setIcon(newIcon);
                            setTimeout(function() {{ marker.setIcon(icon); }}, 1500);
                        }} else if (selectedCluesList.length >= 2) {{
                            alert("Solo se pueden seleccionar 2 unidades para la ruta. Use 'Limpiar Ruta' para empezar de nuevo.");
                        }}
                    }};
                }})(unidad, icon));
            }}
            
            if (currentSelectedClues) setTimeout(function() {{ highlightUnit2D(currentSelectedClues); }}, 1000);
            setTimeout(hideLoader, 2000);
        }} catch(e) {{ console.error(e); hideLoader(); }}
    }}, 10);
}}

function actualizarPanelSeleccion2D() {{
    var panel = document.getElementById("unidadesSeleccionadas");
    var btn = document.getElementById("trazarRutaBtn");
    if (selectedCluesList.length === 0) {{
        panel.innerHTML = '<div style="color:#888;text-align:center;">Selecciona unidades en el mapa 2D</div>';
        btn.disabled = true;
    }} else if (selectedCluesList.length === 1) {{
        var u = unidadesData.find(function(d) {{ return d.clues === selectedCluesList[0]; }});
        panel.innerHTML = '<div class="ruta-item"><div class="ruta-origen-destino">Origen: ' + u.clues + '</div><div>' + u.nombre.substring(0,40) + '</div></div><div style="color:#A57F2C;text-align:center;">Selecciona destino</div>';
        btn.disabled = true;
    }} else {{
        var o = unidadesData.find(function(d) {{ return d.clues === selectedCluesList[0]; }});
        var d = unidadesData.find(function(dd) {{ return dd.clues === selectedCluesList[1]; }});
        panel.innerHTML = '<div class="ruta-item"><div class="ruta-origen-destino">Origen: ' + o.clues + '</div><div>' + o.nombre.substring(0,35) + '</div></div><div class="ruta-item"><div class="ruta-origen-destino">Destino: ' + d.clues + '</div><div>' + d.nombre.substring(0,35) + '</div></div>';
        btn.disabled = false;
    }}
}}

function trazarRuta2D() {{
    if (selectedCluesList.length !== 2) return;
    var o = unidadesData.find(function(d) {{ return d.clues === selectedCluesList[0]; }});
    var d = unidadesData.find(function(dd) {{ return dd.clues === selectedCluesList[1]; }});
    if (!o || !d) return;
    if (routingControl) map2D.removeControl(routingControl);
    routingControl = L.Routing.control({{
        waypoints: [L.latLng(o.lat, o.lng), L.latLng(d.lat, d.lng)],
        routeWhileDragging: false, showAlternatives: false,
        lineOptions: {{ styles: [{{ color: '#4ecdc4', weight: 5, opacity: 0.8 }}] }},
        createMarker: function() {{ return null; }},
        router: L.Routing.osrmv1({{ serviceUrl: 'https://router.project-osrm.org/route/v1' }}),
        show: false
    }}).addTo(map2D);
    routingControl.on('routesfound', function(e) {{
        var route = e.routes[0];
        var dist = (route.summary.totalDistance / 1000).toFixed(1);
        var dur = Math.round(route.summary.totalTime / 60);
        L.popup().setLatLng([(o.lat+d.lat)/2, (o.lng+d.lng)/2]).setContent('<div class="ruta-popup"><div class="distancia">' + dist + ' km</div><div class="tiempo">' + dur + ' minutos</div></div>').openOn(map2D);
        map2D.fitBounds(L.latLngBounds([o.lat, o.lng], [d.lat, d.lng]), {{ padding: [50,50] }});
    }});
}}

function limpiarRuta2D() {{
    selectedCluesList = [];
    if (routingControl) map2D.removeControl(routingControl);
    routingControl = null;
    actualizarPanelSeleccion2D();
    if (map2D) map2D.closePopup();
}}

function toggleModo() {{
    var map3d = document.getElementById("map-3d");
    var map2dDiv = document.getElementById("map-2d");
    var rutasPanel = document.getElementById("rutasPanel");
    var btn = document.querySelector(".btn-toggle");
    modo3D = !modo3D;
    if (modo3D) {{
        showLoader("Cargando mapa 3D...");
        map3d.style.display = "block";
        map2dDiv.style.display = "none";
        rutasPanel.style.display = "none";
        btn.textContent = "Cambiar a Modo 2D (Rutas)";
        selectedCluesList = [];
        if (routingControl) map2D.removeControl(routingControl);
        setTimeout(function() {{
            if (currentSelectedClues) highlightUnit3D(currentSelectedClues);
            setTimeout(hideLoader, 800);
        }}, 100);
    }} else {{
        showLoader("Cargando mapa 2D...");
        map3d.style.display = "none";
        map2dDiv.style.display = "block";
        rutasPanel.style.display = "block";
        btn.textContent = "Cambiar a Modo 3D";
        resetHighlight3D();
        if (map2D) {{
            map2D.invalidateSize();
            if (currentSelectedClues) setTimeout(function() {{ highlightUnit2D(currentSelectedClues); }}, 500);
            setTimeout(hideLoader, 1500);
        }} else {{
            initMap2D();
        }}
    }}
}}

function initMap3D() {{
    deckgl = new deck.DeckGL({{
        container: "map-3d",
        mapStyle: "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
        initialViewState: {{ longitude: {lon_centro}, latitude: {lat_centro}, zoom: 5.5, pitch: 55, bearing: 0 }},
        controller: true,
        getTooltip: function(info) {{
            if (!info.object) return null;
            var d = info.object;
            return {{
                html: '<div style="font-family: Segoe UI, Arial, sans-serif; min-width:260px; padding: 10px;">' +
                    '<div style="font-weight:bold;font-size:13px;margin-bottom:6px;border-bottom: 1px solid rgba(165,127,44,0.3);padding-bottom: 4px;color:#A57F2C;">' + d.nombre_de_la_unidad + '</div>' +
                    '<div style="display: grid; grid-template-columns: 85px 1fr; gap: 4px; font-size: 11px;">' +
                    '<span style="color:#aaa;">CLUES:</span><span style="font-weight:500;">' + d.clues + '</span>' +
                    '<span style="color:#aaa;">Entidad:</span><span>' + d.entidad + '</span>' +
                    '<span style="color:#aaa;">Nivel:</span><span>' + d.nivel + '</span>' +
                    '<span style="color:#aaa;">Indicador:</span><span>' + d.tipo + '</span>' +
                    '<span style="color:#aaa;">Realizado:</span><span style="font-weight:500;">' + d.realizado + '</span>' +
                    '<span style="color:#aaa;">Meta Anual:</span><span>' + d.meta_anual + '</span>' +
                    '<span style="color:#aaa;">Meta Esperada:</span><span>' + d.meta_esperada + '</span>' +
                    '<span style="color:#aaa;">Avance Real:</span><span style="color:' + d.color_hex + '; font-weight:bold;">' + d.avance + '</span>' +
                    '<span style="color:#aaa;">Avance Esperado:</span><span>' + d.pct_dia_fmt + '</span>' +
                    '</div>' +
                    '<div style="margin-top: 6px; padding-top: 4px; border-top: 1px solid rgba(165,127,44,0.2); font-size: 10px; color: #A57F2C;">' + d.estado_semaforo + '</div></div>'
            }};
        }},
        layers: [
            new deck.ColumnLayer({{
                id: "columnas-avance",
                data: unidadesData,
                diskResolution: 12,
                radius: 400,
                extruded: true,
                pickable: true,
                elevationScale: 0.5,
                getPosition: function(d) {{ return [d.lng, d.lat]; }},
                getElevation: function(d) {{ return d.altura; }},
                getFillColor: function(d) {{ return [d.r, d.g, d.b, 200]; }}
            }})
        ]
    }});
    deckgl.canvas.addEventListener('click', function(event) {{
        var pick = deckgl.pickObject({{x: event.clientX, y: event.clientY}});
        if (pick && pick.object) {{
            if (currentSelectedClues === pick.object.clues) {{
                currentSelectedClues = null;
                resetHighlight3D();
            }} else {{
                currentSelectedClues = pick.object.clues;
                highlightUnit3D(currentSelectedClues);
            }}
        }}
    }});
}}

var buscadorInput = document.getElementById("buscadorClues");
var resultadosDiv = document.getElementById("resultadosBusqueda");

function buscarUnidades(texto) {{
    if (texto.length < 2) {{ resultadosDiv.style.display = "none"; return []; }}
    texto = texto.toLowerCase();
    return unidadesData.filter(function(u) {{
        return u.clues.toLowerCase().indexOf(texto) !== -1 || u.nombre.toLowerCase().indexOf(texto) !== -1;
    }}).slice(0,10);
}}

function mostrarResultados(resultados) {{
    resultadosDiv.innerHTML = "";
    if (resultados.length === 0) {{ resultadosDiv.style.display = "none"; return; }}
    for (var i = 0; i < resultados.length; i++) {{
        var u = resultados[i];
        var item = document.createElement("div");
        item.className = "resultado-item";
        item.innerHTML = '<div style="font-weight:bold;color:#A57F2C;font-size:13px;">' + u.clues + '</div><div style="font-size:11px;color:white;">' + u.nombre.substring(0,50) + '</div><div style="font-size:10px;color:#A57F2C;">Avance: ' + u.avance + '</div>';
        item.onclick = (function(u) {{ return function() {{ centrarEnUnidad(u); }}; }})(u);
        resultadosDiv.appendChild(item);
    }}
    resultadosDiv.style.display = "block";
}}

buscadorInput.addEventListener("input", function(e) {{ mostrarResultados(buscarUnidades(e.target.value)); }});
document.addEventListener("click", function(e) {{
    if (!buscadorInput.contains(e.target) && !resultadosDiv.contains(e.target)) resultadosDiv.style.display = "none";
}});

initMap3D();
</script>
</body>
</html>
"""

In [272]:
html_completo = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<title>Hospital Dashboard - Modo 3D y 2D con Geolocalizacion</title>

<script src="https://unpkg.com/deck.gl@latest/dist.min.js"></script>
<script src="https://unpkg.com/maplibre-gl@latest/dist/maplibre-gl.js"></script>
<link href="https://unpkg.com/maplibre-gl@latest/dist/maplibre-gl.css" rel="stylesheet"/>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://unpkg.com/leaflet-routing-machine@latest/dist/leaflet-routing-machine.js"></script>
<link rel="stylesheet" href="https://unpkg.com/leaflet-routing-machine@latest/dist/leaflet-routing-machine.css" />

<style>
html, body {{
    margin: 0;
    width: 100%;
    height: 100%;
    font-family: 'Segoe UI', Arial, sans-serif;
    background: #0a0a0a;
}}

#map-3d {{
    width: 100%;
    height: 100%;
    background: #0d0d0d;
    display: block;
}}

#map-2d {{
    width: 100%;
    height: 100%;
    background: #0d0d0d;
    display: none;
}}

.leaflet-control-attribution {{
    display: none !important;
}}

.loader-overlay {{
    position: fixed;
    top: 0;
    left: 0;
    width: 100%;
    height: 100%;
    background: rgba(0,0,0,0.9);
    backdrop-filter: blur(8px);
    z-index: 20000;
    display: flex;
    align-items: center;
    justify-content: center;
    flex-direction: column;
    opacity: 0;
    visibility: hidden;
    transition: opacity 0.2s ease, visibility 0s linear 0.3s;
    pointer-events: none;
}}

.loader-overlay.active {{
    opacity: 1;
    visibility: visible;
    transition: opacity 0.2s ease, visibility 0s linear 0s;
    pointer-events: all;
}}

.loader-text {{
    color: #A57F2C;
    margin-top: 30px;
    font-size: 14px;
    font-weight: bold;
    letter-spacing: 2px;
    text-transform: uppercase;
    background: rgba(0,0,0,0.6);
    padding: 8px 20px;
    border-radius: 30px;
    border: 1px solid rgba(165,127,44,0.3);
}}

.loader-svg {{
    width: 450px;
    height: 300px;
    animation: spin 5s linear infinite;
    filter: drop-shadow(0 0 15px rgba(165,127,44,0.6));
}}

@keyframes spin {{
    0% {{ transform: rotate(0deg); }}
    100% {{ transform: rotate(360deg); }}
}}

.header {{
    position: fixed;
    top: 0;
    left: 0;
    right: 0;
    z-index: 10000;
    background: #1E5B4F;
    color: white;
    padding: 12px 25px;
    font-size: 18px;
    font-weight: bold;
    text-align: center;
    box-shadow: 0 2px 10px rgba(0,0,0,0.5);
    pointer-events: none;
    border-bottom: 1px solid rgba(165,127,44,0.3);
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 15px;
}}

.header img {{
    height: 40px;
    width: auto;
    filter: brightness(0) invert(1);
}}

.btn-toggle {{
    position: fixed;
    top: 80px;
    right: 20px;
    z-index: 10001;
    background: linear-gradient(135deg, #A57F2C, #1E5B4F);
    color: white;
    border: none;
    padding: 12px 24px;
    border-radius: 30px;
    font-size: 14px;
    font-weight: bold;
    cursor: pointer;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-shadow: 0 4px 15px rgba(0,0,0,0.3);
    transition: all 0.3s;
    pointer-events: auto;
}}

.btn-toggle:hover {{
    transform: translateY(-2px);
    box-shadow: 0 6px 20px rgba(165,127,44,0.4);
}}

.kpi-container {{
    position: fixed;
    top: 70px;
    left: 0;
    right: 0;
    z-index: 10000;
    display: grid;
    grid-template-columns: repeat(5, 1fr);
    gap: 12px;
    padding: 12px 20px;
    pointer-events: none;
}}

.kpi-card {{
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    padding: 10px 15px;
    border-radius: 10px;
    border: 1px solid rgba(165,127,44,0.3);
    pointer-events: auto;
    box-shadow: 0 2px 5px rgba(0,0,0,0.3);
}}

.kpi-label {{
    font-size: 10px;
    color: #A57F2C;
    text-transform: uppercase;
    letter-spacing: 1px;
}}

.kpi-value {{
    font-size: 24px;
    font-weight: bold;
    color: white;
}}

.kpi-sub {{
    font-size: 9px;
    color: #888;
}}

.dashboard-btn {{
    position: fixed;
    top: 380px;
    left: 20px;
    z-index: 10001;
}}

.dashboard-btn button {{
    background: linear-gradient(135deg, #A57F2C, #1E5B4F);
    color: white;
    border: none;
    padding: 8px 18px;
    border-radius: 25px;
    font-size: 12px;
    font-weight: bold;
    cursor: pointer;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-shadow: 0 2px 5px rgba(0,0,0,0.3);
    transition: all 0.3s;
    pointer-events: auto;
}}

.dashboard-btn button:hover {{
    transform: translateY(-1px);
    box-shadow: 0 4px 10px rgba(165,127,44,0.3);
}}

.buscador {{
    position: fixed;
    top: 200px;
    left: 20px;
    z-index: 10001;
    width: 320px;
}}

.buscador-box {{
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    border-radius: 12px;
    padding: 12px;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
}}

.buscador-label {{
    color: #A57F2C;
    font-size: 10px;
    text-transform: uppercase;
    margin-bottom: 8px;
    letter-spacing: 1px;
}}

.buscador-input {{
    width: 100%;
    padding: 10px 15px;
    background: rgba(20,20,20,0.9);
    border: 1px solid rgba(165,127,44,0.3);
    border-radius: 8px;
    color: white;
    font-size: 13px;
    outline: none;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-sizing: border-box;
}}

.buscador-input:focus {{
    border-color: #A57F2C;
    box-shadow: 0 0 5px rgba(165,127,44,0.5);
}}

.resultados {{
    background: rgba(10,10,10,0.95);
    border-radius: 12px;
    margin-top: 8px;
    max-height: 350px;
    overflow-y: auto;
    display: none;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
}}

.resultado-item {{
    padding: 12px 15px;
    cursor: pointer;
    border-bottom: 1px solid rgba(165,127,44,0.1);
    font-family: 'Segoe UI', Arial, sans-serif;
    transition: all 0.2s;
}}

.resultado-item:hover {{
    background: rgba(165,127,44,0.15);
    border-left: 3px solid #A57F2C;
}}

.rutas-panel {{
    position: fixed;
    bottom: 20px;
    left: 20px;
    z-index: 10001;
    background: rgba(10,10,10,0.95);
    backdrop-filter: blur(10px);
    border-radius: 12px;
    min-width: 280px;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    pointer-events: auto;
    padding: 12px;
    display: none;
}}

.rutas-panel h4 {{
    margin: 0 0 10px 0;
    color: #A57F2C;
    font-size: 13px;
    text-align: center;
    border-bottom: 1px solid rgba(165,127,44,0.3);
    padding-bottom: 6px;
}}

.rutas-lista {{
    margin-bottom: 10px;
}}

.ruta-item {{
    background: rgba(30,30,30,0.8);
    border-radius: 8px;
    padding: 8px;
    margin-bottom: 8px;
    font-size: 11px;
    cursor: pointer;
    transition: all 0.2s;
}}

.ruta-item:hover {{
    background: rgba(165,127,44,0.2);
    border-left: 3px solid #4ecdc4;
}}

.ruta-origen-destino {{
    font-weight: bold;
    color: #4ecdc4;
    margin-bottom: 4px;
}}

.ruta-info {{
    color: #aaa;
    font-size: 10px;
    display: flex;
    justify-content: space-between;
}}

.ruta-boton {{
    width: 100%;
    background: linear-gradient(135deg, #1E5B4F, #A57F2C);
    color: white;
    border: none;
    padding: 8px;
    border-radius: 8px;
    font-size: 11px;
    font-weight: bold;
    cursor: pointer;
    margin-top: 5px;
    transition: all 0.2s;
}}

.ruta-boton:hover {{
    transform: translateY(-1px);
    box-shadow: 0 2px 8px rgba(165,127,44,0.4);
}}

.ruta-boton.limpiar {{
    background: rgba(100,100,100,0.6);
    margin-top: 5px;
}}

.ruta-boton.limpiar:hover {{
    background: rgba(150,150,150,0.8);
}}

.seleccion-activa {{
    position: fixed;
    top: 420px;
    left: 20px;
    z-index: 10001;
    background: rgba(165,127,44,0.2);
    backdrop-filter: blur(10px);
    padding: 5px 12px;
    border-radius: 20px;
    border: 1px solid #A57F2C;
    color: #A57F2C;
    font-size: 11px;
    font-weight: bold;
    pointer-events: none;
    display: none;
}}

.semaforo {{
    position: fixed;
    bottom: 20px;
    right: 20px;
    z-index: 10001;
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    padding: 12px 18px;
    border-radius: 12px;
    min-width: 240px;
    border: 1px solid rgba(165,127,44,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    pointer-events: auto;
}}

.footer {{
    position: fixed;
    bottom: 20px;
    left: 320px;
    z-index: 10001;
    background: rgba(10,10,10,0.8);
    backdrop-filter: blur(10px);
    padding: 8px 12px;
    border-radius: 8px;
    font-size: 10px;
    color: #888;
    border: 1px solid rgba(165,127,44,0.2);
    pointer-events: auto;
}}

.deck-tooltip {{
    background: rgba(0,0,0,0.85) !important;
    backdrop-filter: blur(8px) !important;
    border: 1px solid rgba(165,127,44,0.4) !important;
    border-radius: 8px !important;
    box-shadow: 0 4px 15px rgba(0,0,0,0.5) !important;
    padding: 0 !important;
}}

.leaflet-popup-content-wrapper {{
    background: rgba(10,10,10,0.95);
    backdrop-filter: blur(10px);
    border: 1px solid rgba(165,127,44,0.3);
    border-radius: 12px;
    color: white;
    min-width: 200px;
}}

.leaflet-popup-tip {{
    background: rgba(10,10,10,0.95);
    border: 1px solid rgba(165,127,44,0.3);
}}

.leaflet-popup-content {{
    margin: 8px 12px;
    line-height: 1.4;
}}

.ruta-popup {{
    font-family: 'Segoe UI', Arial, sans-serif;
    text-align: center;
}}

.ruta-popup .distancia {{
    font-size: 16px;
    font-weight: bold;
    color: #4ecdc4;
    margin-bottom: 5px;
}}

.ruta-popup .tiempo {{
    font-size: 13px;
    color: #A57F2C;
}}

.custom-marker {{
    background: transparent;
    border: none;
}}

.custom-marker div {{
    width: 6px;
    height: 6px;
    border-radius: 50%;
    border: 1px solid rgba(0,0,0,0.3);
    box-shadow: 0 0 2px rgba(0,0,0,0.5);
    transition: all 0.2s ease;
}}

.custom-marker div:hover {{
    transform: scale(1.5);
    box-shadow: 0 0 4px rgba(165,127,44,0.8);
}}

.selected-marker div {{
    width: 16px !important;
    height: 16px !important;
    border: 3px solid #4ecdc4 !important;
    box-shadow: 0 0 15px rgba(78, 205, 196, 0.8) !important;
    animation: pulse 1.5s ease-in-out infinite !important;
}}

@keyframes pulse {{
    0% {{ transform: scale(1); opacity: 1; }}
    50% {{ transform: scale(1.3); opacity: 0.7; }}
    100% {{ transform: scale(1); opacity: 1; }}
}}

.leaflet-routing-container {{
    display: none !important;
}}

/* Boton de geolocalizacion */
.btn-geoloc {{
    position: fixed;
    top: 280px;
    left: 20px;
    z-index: 10001;
    background: linear-gradient(135deg, #1E5B4F, #A57F2C);
    color: white;
    border: none;
    padding: 12px 20px;
    border-radius: 30px;
    font-size: 13px;
    font-weight: bold;
    cursor: pointer;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-shadow: 0 4px 15px rgba(0,0,0,0.3);
    transition: all 0.3s;
    pointer-events: auto;
    width: 280px;
    text-align: center;
}}

.btn-geoloc:hover {{
    transform: translateY(-2px);
    box-shadow: 0 6px 20px rgba(165,127,44,0.4);
}}

.btn-geoloc.loading {{
    opacity: 0.7;
    cursor: wait;
}}
</style>
</head>
<body>

<div id="map-3d"></div>
<div id="map-2d"></div>

<div id="loaderOverlay" class="loader-overlay">
    <img class="loader-svg" src="{svg_data_uri}" alt="Cargando...">
    <div class="loader-text">Cargando mapa...</div>
</div>

<button class="btn-toggle" onclick="toggleModo()">Cambiar a Modo 2D (Rutas)</button>

<div class="header">
    <img src="https://imssbienestar.gob.mx/assets/img/imb_b.svg" 
         alt="IMSS Bienestar" 
         onerror="this.style.display='none'">
    Hospital Dashboard - Productividad por Unidad Medica
</div>

<div class="kpi-container">
    <div class="kpi-card">
        <div class="kpi-label">Unidades Activas</div>
        <div class="kpi-value">{total_unidades_activas:,}</div>
        <div class="kpi-sub">Cirugia: {unidades_cirugias_activas} | Consulta: {unidades_consultas_activas}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Avance General</div>
        <div class="kpi-value">{avg_avance_general:.1f}%</div>
        <div class="kpi-sub">Cirugia: {avg_avance_cirugias:.1f}% | Consulta: {avg_avance_consultas:.1f}%</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Volumen Realizado - Cirugia</div>
        <div class="kpi-value" style="color: #ffffff;">{volumen_cirugias:,.0f}</div>
        <div class="kpi-sub">Total de cirugias realizadas</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Volumen Realizado - Consulta</div>
        <div class="kpi-value" style="color: #4ecdc4;">{volumen_consultas:,.0f}</div>
        <div class="kpi-sub" style="font-size: 8px;">General: {consulta_general:,.0f} | Especialidad: {consulta_especialidad:,.0f}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Cobertura</div>
        <div class="kpi-value">{entidades_unicas}</div>
        <div class="kpi-sub">Entidades Federativas</div>
    </div>
</div>

<div class="dashboard-btn">
    <button onclick="window.open('https://argontc.shinyapps.io/pptx/', '_blank')">
        Ver Dashboard de Reportes
    </button>
</div>

<div class="buscador">
    <div class="buscador-box">
        <div class="buscador-label">Buscador de Unidades</div>
        <input type="text" id="buscadorClues" class="buscador-input" placeholder="Buscar por CLUES o nombre...">
    </div>
    <div id="resultadosBusqueda" class="resultados"></div>
</div>

<div id="seleccionInfo" class="seleccion-activa"></div>

<div class="rutas-panel" id="rutasPanel">
    <h4>Trazado de Rutas (Modo 2D)</h4>
    <div id="unidadesSeleccionadas" class="rutas-lista">
        <div style="color: #888; font-size: 11px; text-align: center;">Selecciona unidades en el mapa 2D</div>
    </div>
    <button id="trazarRutaBtn" class="ruta-boton" onclick="trazarRuta2D()" disabled>Trazar Ruta</button>
    <button id="limpiarRutaBtn" class="ruta-boton limpiar" onclick="limpiarRuta2D()">Limpiar Ruta</button>
</div>

<button id="btnGeoloc" class="btn-geoloc" onclick="obtenerMiUbicacion()">
    Mi Ubicacion - Ruta a CLUES mas cercana
</button>

<div class="semaforo">
    <div style="color: white; font-size: 13px; font-weight: bold; margin-bottom: 10px; text-align: center; border-bottom: 1px solid rgba(165,127,44,0.3); padding-bottom: 5px;">
        Semaforo de Avance
    </div>
    <div style="font-size: 11px; color: #A57F2C; margin-bottom: 10px; text-align: center; background: rgba(0,0,0,0.5); padding: 5px; border-radius: 6px;">
        Ano transcurrido: <strong>{pct_dia_esperado:.1f}%</strong>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['verde fuerte']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Verde Fuerte: Avance >= {verde_fuerte_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['verde claro']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Verde Claro: Avance >= {verde_claro_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['amarillo']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Amarillo: Avance >= {amarillo_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['rojo']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Rojo: Avance < {amarillo_umbral:.1f}%</span>
    </div>
</div>

<div class="footer">
    Datos al corte | Avance basado en meta esperada al dia de hoy
</div>

<script>
var unidadesData = {unidades_data_json};
var modo3D = true;
var selectedCluesList = [];
var map2D = null;
var routingControl = null;
var deckgl = null;
var loaderTimeout = null;
var mapLoadingStarted = false;
var currentSelectedClues = null;
var currentSelectedMarker = null;
var userLocationMarker = null;
var currentGeolocRouting = null;

function showLoader(texto) {{
    var loader = document.getElementById("loaderOverlay");
    var loaderText = document.querySelector(".loader-text");
    if (texto) loaderText.textContent = texto;
    mapLoadingStarted = true;
    void loader.offsetWidth;
    loader.classList.add("active");
    if (loaderTimeout) clearTimeout(loaderTimeout);
    loaderTimeout = setTimeout(function() {{ hideLoader(); }}, 5000);
}}

function hideLoader() {{
    var loader = document.getElementById("loaderOverlay");
    loader.classList.remove("active");
    if (loaderTimeout) {{ clearTimeout(loaderTimeout); loaderTimeout = null; }}
    mapLoadingStarted = false;
}}

function highlightUnit3D(clues) {{
    if (!deckgl) return;
    var selectedUnit = unidadesData.find(u => u.clues === clues);
    if (!selectedUnit) return;
    
    var capaNeon = [
        new deck.ScatterplotLayer({{
            id: "neon-glow",
            data: [selectedUnit],
            getPosition: function(d) {{ return [d.lng, d.lat]; }},
            radiusScale: 1,
            radiusMinPixels: 30,
            radiusMaxPixels: 60,
            getRadius: 400,
            getFillColor: [78, 205, 196, 80],
            stroked: true,
            getLineColor: [78, 205, 196, 200],
            getLineWidth: 3,
            pickable: false
        }}),
        new deck.ScatterplotLayer({{
            id: "neon-core",
            data: [selectedUnit],
            getPosition: function(d) {{ return [d.lng, d.lat]; }},
            radiusScale: 1,
            radiusMinPixels: 15,
            radiusMaxPixels: 30,
            getRadius: 200,
            getFillColor: [78, 205, 196, 180],
            stroked: false,
            pickable: false
        }})
    ];
    
    deckgl.setProps({{
        layers: [
            new deck.ColumnLayer({{
                id: "columnas-avance",
                data: unidadesData,
                diskResolution: 12,
                radius: 400,
                extruded: true,
                pickable: true,
                elevationScale: 0.5,
                getPosition: function(d) {{ return [d.lng, d.lat]; }},
                getElevation: function(d) {{ return d.altura; }},
                getFillColor: function(d) {{
                    if (d.clues === clues) return [255, 215, 0, 255];
                    return [d.r, d.g, d.b, 200];
                }},
                getLineColor: function(d) {{
                    if (d.clues === clues) return [255, 255, 255, 255];
                    return [0, 0, 0, 0];
                }},
                getLineWidth: function(d) {{
                    if (d.clues === clues) return 4;
                    return 0;
                }}
            }}),
            ...capaNeon
        ]
    }});
}}

function resetHighlight3D() {{
    if (!deckgl) return;
    deckgl.setProps({{
        layers: [
            new deck.ColumnLayer({{
                id: "columnas-avance",
                data: unidadesData,
                diskResolution: 12,
                radius: 400,
                extruded: true,
                pickable: true,
                elevationScale: 0.5,
                getPosition: function(d) {{ return [d.lng, d.lat]; }},
                getElevation: function(d) {{ return d.altura; }},
                getFillColor: function(d) {{ return [d.r, d.g, d.b, 200]; }}
            }})
        ]
    }});
}}

function centrarEnUnidad(unidad) {{
    currentSelectedClues = unidad.clues;
    if (modo3D && deckgl) {{
        deckgl.setProps({{ 
            initialViewState: {{ 
                longitude: unidad.lng, latitude: unidad.lat, zoom: 14, pitch: 60, bearing: 0,
                transitionDuration: 1500, transitionInterpolator: new deck.FlyToInterpolator({{ speed: 1.2 }})
            }}
        }});
        setTimeout(function() {{ highlightUnit3D(unidad.clues); }}, 1500);
    }} else if (map2D) {{
        map2D.setView([unidad.lat, unidad.lng], 14);
        setTimeout(function() {{ highlightUnit2D(unidad.clues); }}, 500);
    }}
    buscadorInput.value = unidad.clues;
    resultadosDiv.style.display = "none";
    var seleccionDiv = document.getElementById("seleccionInfo");
    seleccionDiv.innerHTML = "Seleccionado: " + unidad.clues + " - " + unidad.nombre.substring(0, 35);
    seleccionDiv.style.display = "block";
    setTimeout(function() {{ seleccionDiv.style.display = "none"; }}, 3000);
}}

function highlightUnit2D(clues) {{
    if (!map2D) return;
    map2D.eachLayer(function(layer) {{
        if (layer instanceof L.Marker && layer.getPopup()) {{
            var content = layer.getPopup().getContent();
            var match = content.match(/CLUES: (\\S+)/);
            if (match && match[1] === clues) {{
                if (currentSelectedMarker) {{
                    var origIcon = L.divIcon({{
                        className: 'custom-marker',
                        html: '<div style="width: 6px; height: 6px; background-color: ' + currentSelectedMarker.color + '; border-radius: 50%; border: 1px solid #A57F2C;"></div>',
                        iconSize: [6, 6], iconAnchor: [3, 3], popupAnchor: [0, -3]
                    }});
                    currentSelectedMarker.setIcon(origIcon);
                }}
                var unidad = unidadesData.find(u => u.clues === clues);
                var highlightedIcon = L.divIcon({{
                    className: 'custom-marker selected-marker',
                    html: '<div style="width: 16px; height: 16px; background-color: ' + unidad.color_hex + '; border-radius: 50%; border: 3px solid #4ecdc4; box-shadow: 0 0 15px rgba(78,205,196,0.8);"></div>',
                    iconSize: [16, 16], iconAnchor: [8, 8], popupAnchor: [0, -8]
                }});
                layer.setIcon(highlightedIcon);
                currentSelectedMarker = layer;
                currentSelectedMarker.color = unidad.color_hex;
                layer.openPopup();
            }}
        }}
    }});
}}

function trazarRutaEntreDosPuntos(origenLat, origenLng, destinoLat, destinoLng, mostrarPopupCentro = true) {{
    if (routingControl) map2D.removeControl(routingControl);
    
    routingControl = L.Routing.control({{
        waypoints: [L.latLng(origenLat, origenLng), L.latLng(destinoLat, destinoLng)],
        routeWhileDragging: false,
        showAlternatives: false,
        lineOptions: {{ styles: [{{ color: '#4ecdc4', weight: 5, opacity: 0.8 }}] }},
        createMarker: function() {{ return null; }},
        router: L.Routing.osrmv1({{ serviceUrl: 'https://router.project-osrm.org/route/v1' }}),
        show: false
    }}).addTo(map2D);
    
    if (mostrarPopupCentro) {{
        routingControl.on('routesfound', function(e) {{
            var route = e.routes[0];
            var dist = (route.summary.totalDistance / 1000).toFixed(1);
            var dur = Math.round(route.summary.totalTime / 60);
            L.popup()
                .setLatLng([(origenLat + destinoLat)/2, (origenLng + destinoLng)/2])
                .setContent('<div class="ruta-popup"><div class="distancia">' + dist + ' km</div><div class="tiempo">' + dur + ' minutos</div></div>')
                .openOn(map2D);
        }});
    }}
    
    return routingControl;
}}

function obtenerCluesMasCercana(lat, lng) {{
    var masCercana = null;
    var distanciaMinima = Infinity;
    
    for (var i = 0; i < unidadesData.length; i++) {{
        var unidad = unidadesData[i];
        var R = 6371;
        var dLat = (unidad.lat - lat) * Math.PI / 180;
        var dLon = (unidad.lng - lng) * Math.PI / 180;
        var a = Math.sin(dLat/2) * Math.sin(dLat/2) +
                Math.cos(lat * Math.PI / 180) * Math.cos(unidad.lat * Math.PI / 180) *
                Math.sin(dLon/2) * Math.sin(dLon/2);
        var c = 2 * Math.atan2(Math.sqrt(a), Math.sqrt(1-a));
        var distancia = R * c;
        
        if (distancia < distanciaMinima) {{
            distanciaMinima = distancia;
            masCercana = unidad;
        }}
    }}
    
    return {{ unidad: masCercana, distancia: distanciaMinima }};
}}

function obtenerMiUbicacion() {{
    var btn = document.getElementById("btnGeoloc");
    
    if (!navigator.geolocation) {{
        alert("Su navegador no soporta geolocalizacion");
        return;
    }}
    
    btn.classList.add("loading");
    btn.textContent = "Obteniendo ubicacion...";
    
    navigator.geolocation.getCurrentPosition(
        function(position) {{
            var lat = position.coords.latitude;
            var lng = position.coords.longitude;
            
            btn.classList.remove("loading");
            btn.textContent = "Mi Ubicacion - Ruta a CLUES mas cercana";
            
            if (!modo3D && map2D) {{
                if (userLocationMarker) {{
                    map2D.removeLayer(userLocationMarker);
                }}
                
                var userIcon = L.divIcon({{
                    className: 'custom-marker',
                    html: '<div style="width: 14px; height: 14px; background-color: #4ecdc4; border-radius: 50%; border: 2px solid white; box-shadow: 0 0 10px rgba(78,205,196,0.8);"></div>',
                    iconSize: [14, 14],
                    iconAnchor: [7, 7]
                }});
                
                userLocationMarker = L.marker([lat, lng], {{ icon: userIcon }}).addTo(map2D);
                userLocationMarker.bindPopup('<strong>Tu ubicacion</strong><br>Lat: ' + lat.toFixed(6) + '<br>Lng: ' + lng.toFixed(6)).openPopup();
                
                var cercana = obtenerCluesMasCercana(lat, lng);
                
                if (cercana.unidad) {{
                    var distanciaTexto = cercana.distancia < 1 ? 
                        Math.round(cercana.distancia * 1000) + ' metros' : 
                        cercana.distancia.toFixed(2) + ' km';
                    
                    L.popup()
                        .setLatLng([cercana.unidad.lat, cercana.unidad.lng])
                        .setContent('<div class="ruta-popup"><strong style="color:#A57F2C;">CLUES mas cercana</strong><br>' +
                            '<strong>' + cercana.unidad.clues + '</strong><br>' +
                            cercana.unidad.nombre.substring(0, 40) + '<br>' +
                            '<span style="color:#4ecdc4;">Distancia: ' + distanciaTexto + '</span></div>')
                        .openOn(map2D);
                    
                    if (currentGeolocRouting) {{
                        map2D.removeControl(currentGeolocRouting);
                    }}
                    
                    currentGeolocRouting = trazarRutaEntreDosPuntos(lat, lng, cercana.unidad.lat, cercana.unidad.lng, true);
                    
                    map2D.setView([lat, lng], 13);
                    
                    setTimeout(function() {{
                        var bounds = L.latLngBounds([lat, lng], [cercana.unidad.lat, cercana.unidad.lng]);
                        map2D.fitBounds(bounds, {{ padding: [50, 50] }});
                    }}, 500);
                }}
            }} else {{
                alert("Cambie al modo 2D para usar la geolocalizacion y ver rutas");
                btn.classList.remove("loading");
                btn.textContent = "Mi Ubicacion - Ruta a CLUES mas cercana";
            }}
        }},
        function(error) {{
            btn.classList.remove("loading");
            btn.textContent = "Mi Ubicacion - Ruta a CLUES mas cercana";
            
            var mensaje = "";
            switch(error.code) {{
                case error.PERMISSION_DENIED:
                    mensaje = "Permiso denegado para acceder a su ubicacion";
                    break;
                case error.POSITION_UNAVAILABLE:
                    mensaje = "No se pudo obtener su ubicacion";
                    break;
                case error.TIMEOUT:
                    mensaje = "Tiempo de espera agotado";
                    break;
                default:
                    mensaje = "Error al obtener ubicacion";
                    break;
            }}
            alert("Error de geolocalizacion: " + mensaje);
        }},
        {{
            enableHighAccuracy: true,
            timeout: 10000,
            maximumAge: 0
        }}
    );
}}

function initMap2D() {{
    showLoader("Cargando mapa 2D...");
    setTimeout(function() {{
        try {{
            if (map2D) map2D.remove();
            map2D = L.map('map-2d', {{ attributionControl: false }}).setView([{lat_centro}, {lon_centro}], 6);
            L.tileLayer('https://{{s}}.basemaps.cartocdn.com/dark_all/{{z}}/{{x}}/{{y}}.png', {{
                subdomains: 'abcd', minZoom: 4, maxZoom: 18, attribution: ''
            }}).addTo(map2D);
            
            for (var i = 0; i < unidadesData.length; i++) {{
                var unidad = unidadesData[i];
                var icon = L.divIcon({{
                    className: 'custom-marker',
                    html: '<div style="width: 6px; height: 6px; background-color: ' + unidad.color_hex + '; border-radius: 50%; border: 1px solid #A57F2C;"></div>',
                    iconSize: [6, 6], iconAnchor: [3, 3], popupAnchor: [0, -3]
                }});
                var marker = L.marker([unidad.lat, unidad.lng], {{ icon: icon }}).addTo(map2D);
                marker.bindPopup('<div style="font-family: Segoe UI, Arial, sans-serif; min-width: 180px;">' +
                    '<strong style="color: #A57F2C;">' + unidad.nombre_de_la_unidad + '</strong><br>' +
                    '<span style="font-size: 11px;">CLUES: ' + unidad.clues + '</span><br>' +
                    '<span style="font-size: 11px;">Avance Real: <span style="color:' + unidad.color_hex + '; font-weight:bold;">' + unidad.avance + '</span></span><br>' +
                    '<span style="font-size: 11px;">Avance Esperado: ' + unidad.pct_dia_fmt + '</span><br>' +
                    '<span style="font-size: 10px; color: #888;">' + unidad.estado_semaforo + '</span></div>');
                marker.on('click', (function(unidad, icon) {{
                    return function(e) {{
                        L.DomEvent.stopPropagation(e);
                        if (selectedCluesList.length < 2 && selectedCluesList.indexOf(unidad.clues) === -1) {{
                            selectedCluesList.push(unidad.clues);
                            actualizarPanelSeleccion2D();
                            var newIcon = L.divIcon({{
                                className: 'custom-marker',
                                html: '<div style="width: 10px; height: 10px; background-color: ' + unidad.color_hex + '; border-radius: 50%; border: 2px solid #4ecdc4;"></div>',
                                iconSize: [10, 10], iconAnchor: [5, 5], popupAnchor: [0, -5]
                            }});
                            marker.setIcon(newIcon);
                            setTimeout(function() {{ marker.setIcon(icon); }}, 1500);
                        }} else if (selectedCluesList.length >= 2) {{
                            alert("Solo se pueden seleccionar 2 unidades para la ruta. Use 'Limpiar Ruta' para empezar de nuevo.");
                        }}
                    }};
                }})(unidad, icon));
            }}
            
            if (currentSelectedClues) setTimeout(function() {{ highlightUnit2D(currentSelectedClues); }}, 1000);
            setTimeout(hideLoader, 2000);
        }} catch(e) {{ console.error(e); hideLoader(); }}
    }}, 10);
}}

function actualizarPanelSeleccion2D() {{
    var panel = document.getElementById("unidadesSeleccionadas");
    var btn = document.getElementById("trazarRutaBtn");
    if (selectedCluesList.length === 0) {{
        panel.innerHTML = '<div style="color:#888;text-align:center;">Selecciona unidades en el mapa 2D</div>';
        btn.disabled = true;
    }} else if (selectedCluesList.length === 1) {{
        var u = unidadesData.find(function(d) {{ return d.clues === selectedCluesList[0]; }});
        panel.innerHTML = '<div class="ruta-item"><div class="ruta-origen-destino">Origen: ' + u.clues + '</div><div>' + u.nombre.substring(0,40) + '</div></div><div style="color:#A57F2C;text-align:center;">Selecciona destino</div>';
        btn.disabled = true;
    }} else {{
        var o = unidadesData.find(function(d) {{ return d.clues === selectedCluesList[0]; }});
        var d = unidadesData.find(function(dd) {{ return dd.clues === selectedCluesList[1]; }});
        panel.innerHTML = '<div class="ruta-item"><div class="ruta-origen-destino">Origen: ' + o.clues + '</div><div>' + o.nombre.substring(0,35) + '</div></div><div class="ruta-item"><div class="ruta-origen-destino">Destino: ' + d.clues + '</div><div>' + d.nombre.substring(0,35) + '</div></div>';
        btn.disabled = false;
    }}
}}

function trazarRuta2D() {{
    if (selectedCluesList.length !== 2) return;
    var o = unidadesData.find(function(d) {{ return d.clues === selectedCluesList[0]; }});
    var d = unidadesData.find(function(dd) {{ return dd.clues === selectedCluesList[1]; }});
    if (!o || !d) return;
    trazarRutaEntreDosPuntos(o.lat, o.lng, d.lat, d.lng, true);
}}

function limpiarRuta2D() {{
    selectedCluesList = [];
    if (routingControl) map2D.removeControl(routingControl);
    routingControl = null;
    if (currentGeolocRouting) {{
        map2D.removeControl(currentGeolocRouting);
        currentGeolocRouting = null;
    }}
    actualizarPanelSeleccion2D();
    if (map2D) map2D.closePopup();
}}

function toggleModo() {{
    var map3d = document.getElementById("map-3d");
    var map2dDiv = document.getElementById("map-2d");
    var rutasPanel = document.getElementById("rutasPanel");
    var btn = document.querySelector(".btn-toggle");
    modo3D = !modo3D;
    if (modo3D) {{
        showLoader("Cargando mapa 3D...");
        map3d.style.display = "block";
        map2dDiv.style.display = "none";
        rutasPanel.style.display = "none";
        btn.textContent = "Cambiar a Modo 2D (Rutas)";
        selectedCluesList = [];
        if (routingControl) map2D.removeControl(routingControl);
        setTimeout(function() {{
            if (currentSelectedClues) highlightUnit3D(currentSelectedClues);
            setTimeout(hideLoader, 800);
        }}, 100);
    }} else {{
        showLoader("Cargando mapa 2D...");
        map3d.style.display = "none";
        map2dDiv.style.display = "block";
        rutasPanel.style.display = "block";
        btn.textContent = "Cambiar a Modo 3D";
        resetHighlight3D();
        if (map2D) {{
            map2D.invalidateSize();
            if (currentSelectedClues) setTimeout(function() {{ highlightUnit2D(currentSelectedClues); }}, 500);
            setTimeout(hideLoader, 1500);
        }} else {{
            initMap2D();
        }}
    }}
}}

function initMap3D() {{
    deckgl = new deck.DeckGL({{
        container: "map-3d",
        mapStyle: "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
        initialViewState: {{ longitude: {lon_centro}, latitude: {lat_centro}, zoom: 5.5, pitch: 55, bearing: 0 }},
        controller: true,
        getTooltip: function(info) {{
            if (!info.object) return null;
            var d = info.object;
            return {{
                html: '<div style="font-family: Segoe UI, Arial, sans-serif; min-width:260px; padding: 10px;">' +
                    '<div style="font-weight:bold;font-size:13px;margin-bottom:6px;border-bottom: 1px solid rgba(165,127,44,0.3);padding-bottom: 4px;color:#A57F2C;">' + d.nombre_de_la_unidad + '</div>' +
                    '<div style="display: grid; grid-template-columns: 85px 1fr; gap: 4px; font-size: 11px;">' +
                    '<span style="color:#aaa;">CLUES:</span><span style="font-weight:500;">' + d.clues + '</span>' +
                    '<span style="color:#aaa;">Entidad:</span><span>' + d.entidad + '</span>' +
                    '<span style="color:#aaa;">Nivel:</span><span>' + d.nivel + '</span>' +
                    '<span style="color:#aaa;">Indicador:</span><span>' + d.tipo + '</span>' +
                    '<span style="color:#aaa;">Realizado:</span><span style="font-weight:500;">' + d.realizado + '</span>' +
                    '<span style="color:#aaa;">Meta Anual:</span><span>' + d.meta_anual + '</span>' +
                    '<span style="color:#aaa;">Meta Esperada:</span><span>' + d.meta_esperada + '</span>' +
                    '<span style="color:#aaa;">Avance Real:</span><span style="color:' + d.color_hex + '; font-weight:bold;">' + d.avance + '</span>' +
                    '<span style="color:#aaa;">Avance Esperado:</span><span>' + d.pct_dia_fmt + '</span>' +
                    '</div>' +
                    '<div style="margin-top: 6px; padding-top: 4px; border-top: 1px solid rgba(165,127,44,0.2); font-size: 10px; color: #A57F2C;">' + d.estado_semaforo + '</div></div>'
            }};
        }},
        layers: [
            new deck.ColumnLayer({{
                id: "columnas-avance",
                data: unidadesData,
                diskResolution: 12,
                radius: 400,
                extruded: true,
                pickable: true,
                elevationScale: 0.5,
                getPosition: function(d) {{ return [d.lng, d.lat]; }},
                getElevation: function(d) {{ return d.altura; }},
                getFillColor: function(d) {{ return [d.r, d.g, d.b, 200]; }}
            }})
        ]
    }});
    deckgl.canvas.addEventListener('click', function(event) {{
        var pick = deckgl.pickObject({{x: event.clientX, y: event.clientY}});
        if (pick && pick.object) {{
            if (currentSelectedClues === pick.object.clues) {{
                currentSelectedClues = null;
                resetHighlight3D();
            }} else {{
                currentSelectedClues = pick.object.clues;
                highlightUnit3D(currentSelectedClues);
            }}
        }}
    }});
}}

var buscadorInput = document.getElementById("buscadorClues");
var resultadosDiv = document.getElementById("resultadosBusqueda");

function buscarUnidades(texto) {{
    if (texto.length < 2) {{ resultadosDiv.style.display = "none"; return []; }}
    texto = texto.toLowerCase();
    return unidadesData.filter(function(u) {{
        return u.clues.toLowerCase().indexOf(texto) !== -1 || u.nombre.toLowerCase().indexOf(texto) !== -1;
    }}).slice(0,10);
}}

function mostrarResultados(resultados) {{
    resultadosDiv.innerHTML = "";
    if (resultados.length === 0) {{ resultadosDiv.style.display = "none"; return; }}
    for (var i = 0; i < resultados.length; i++) {{
        var u = resultados[i];
        var item = document.createElement("div");
        item.className = "resultado-item";
        item.innerHTML = '<div style="font-weight:bold;color:#A57F2C;font-size:13px;">' + u.clues + '</div><div style="font-size:11px;color:white;">' + u.nombre.substring(0,50) + '</div><div style="font-size:10px;color:#A57F2C;">Avance: ' + u.avance + '</div>';
        item.onclick = (function(u) {{ return function() {{ centrarEnUnidad(u); }}; }})(u);
        resultadosDiv.appendChild(item);
    }}
    resultadosDiv.style.display = "block";
}}

buscadorInput.addEventListener("input", function(e) {{ mostrarResultados(buscarUnidades(e.target.value)); }});
document.addEventListener("click", function(e) {{
    if (!buscadorInput.contains(e.target) && !resultadosDiv.contains(e.target)) resultadosDiv.style.display = "none";
}});

initMap3D();
</script>
</body>
</html>
"""

In [273]:
archivo_salida = Path("index.html")

with open(archivo_salida, "w", encoding="utf-8") as f:
    f.write(html_completo)

webbrowser.open(archivo_salida.resolve().as_uri())

True